# Pipeline v16_1 -- v13.6 inference + v16 LoRA

**EXACT 2026 -- XAI Challenge @ IJCNN | Kaggle T4x2**

A/B comparison notebook. Pipeline is IDENTICAL to the other v1.1 sibling --
only the LoRA adapter differs.

## What this measures
- LoRA used: `v16` (`/kaggle/working/qwen3_cot_lora_v16`)
- Eval pipeline: declarative Pass-2 -> Z3 -> Stage C.5 repair -> rule-based decide
- Metrics: accuracy + macro-F1 + weighted-F1 + per-class F1 + No-F1 + Unknown-F1 + q_idx breakdown + confusion matrix
- Compare: load `pipeline_results_16_1.json` from both runs side by side.

## NO LGBM
LightGBM router removed from final path per reviewer note #1. Kept as ablation cell only.


In [1]:
#!/usr/bin/env python3
"""
notebook_v13_4_with_lora.py -- EXACT 2026 v13.4
Same architecture as v13.3 (full comparison) but Stage D (Qwen-CoT)
uses LoRA v14 trained on dataset's `explanation` field.
"""


"\nnotebook_v13_4_with_lora.py -- EXACT 2026 v13.4\nSame architecture as v13.3 (full comparison) but Stage D (Qwen-CoT)\nuses LoRA v14 trained on dataset's `explanation` field.\n"

In [2]:
# Fix Kaggle T4
import os
os.environ['VLLM_USE_V1'] = '0'
os.environ['VLLM_WORKER_MULTIPROC_METHOD'] = 'spawn'
os.environ['VLLM_ATTENTION_BACKEND'] = 'FLASH_ATTN'


In [3]:
# ==================================================================
# STAGE 0 -- Install Dependencies & Config
# FIX v2: pin transformers TRUOC khi install vLLM.
# Kaggle torch 2.10.0+cu128 + transformers 4.51+ gay 2 loi khac nhau.
# Strategy: thu vllm pre-installed -> pin transformers -> thu tung pair.
# NOTE: Cell nay PHẢI chạy trước mọi import torch/vllm trong kernel sạch.
# ==================================================================
import subprocess, sys, os, importlib, shutil, glob, json, re, time, csv, traceback
from pathlib import Path
from dataclasses import dataclass, field

os.environ['VLLM_USE_V1'] = '0'
os.environ['VLLM_WORKER_MULTIPROC_METHOD'] = 'spawn'
os.environ['VLLM_CONFIGURE_LOGGING'] = '1'
os.environ['VLLM_LOGGING_LEVEL'] = 'WARNING'


def _pip(*args):
    cmd = [sys.executable, "-m", "pip", "install", "--quiet", "--break-system-packages"] + list(args)
    p = subprocess.run(cmd, capture_output=True, text=True)
    if p.returncode != 0:
        print("\n[PIP FAIL]", " ".join(cmd), flush=True)
        print(p.stdout[-2000:], flush=True)
        print(p.stderr[-3000:], flush=True)
    return p


def _clear_vllm():
    for mod in list(sys.modules.keys()):
        if any(x in mod for x in ("vllm", "transformers")):
            del sys.modules[mod]


def _try_import():
    try:
        from vllm import LLM, SamplingParams
        return True, ""
    except Exception as e:
        return False, str(e).split("\n")[0][:140]

print("=" * 80, flush=True)
print("STAGE 0 vLLM compatibility installer", flush=True)
print("IMPORTANT: if this cell imported torch/vllm failed before, restart kernel and run from top.", flush=True)
print("=" * 80, flush=True)

# ---- Step 1: check pre-installed vllm (Kaggle may already have one) ----
ok, msg = _try_import()
if ok:
    print("Pre-installed vLLM: OK (no vLLM install needed)", flush=True)
    _vllm_ok = True
else:
    print(f"Pre-installed vLLM: FAIL ({msg})", flush=True)

    # ---- Step 2: try (transformers pin) + (vllm version) pairs ----
    PAIRS = [
        # (transformers, vllm) -- ordered from likely to older fallbacks
        ("4.48.0", ""),             # latest vllm + safer transformers
        ("4.46.3", "0.9.1"),
        ("4.46.3", "0.8.5.post1"),
        ("4.48.0", "0.9.1"),
        ("4.44.2", "0.8.5.post1"),
        ("4.44.2", "0.7.3"),
        ("4.46.3", "0.6.6.post1"),
        ("4.46.3", "0.6.6"),
    ]
    _vllm_ok = False
    for tv, vv in PAIRS:
        vllm_pkg = f"vllm=={vv}" if vv else "vllm"
        print(f"  Trying transformers=={tv} + {vllm_pkg}...", end=" ", flush=True)
        p1 = _pip(f"transformers=={tv}")
        p2 = _pip(vllm_pkg)
        _clear_vllm()
        ok, msg = _try_import()
        if ok:
            print("OK", flush=True)
            _vllm_ok = True
            break
        else:
            print(f"FAIL ({msg})", flush=True)

    if not _vllm_ok:
        raise RuntimeError(
            "No vLLM+transformers pair worked. "
            "Recommended fallback: use the transformers_fallback notebook, or restart Kaggle session and retry."
        )

# ---- z3-solver: ensure installed regardless of vLLM path ----
try:
    import z3  # noqa
    print("z3-solver: already OK", flush=True)
except Exception:
    print("Installing z3-solver...", flush=True)
    _pip("z3-solver")
    import z3  # noqa
    print("z3-solver: OK", flush=True)

# ---- Kill flashinfer if present (can cause CUDA errors on Kaggle) ----
for d in glob.glob("/usr/local/lib/python3.12/dist-packages/flashinfer*"):
    tgt = d + "_DISABLED"
    if os.path.exists(d) and not os.path.exists(tgt):
        try:
            os.rename(d, tgt)
        except Exception:
            shutil.rmtree(d, ignore_errors=True)
shutil.rmtree("/root/.cache/flashinfer", ignore_errors=True)
for mod in list(sys.modules.keys()):
    if "flashinfer" in mod:
        del sys.modules[mod]
os.environ["VLLM_ATTENTION_BACKEND"] = "FLASH_ATTN"

# ---- Final imports ----
import torch
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer
from z3 import (
    Int, IntSort, IntVal, BoolSort, Function,
    ForAll, Exists, And, Or, Not, Implies, Solver, sat, unsat,
)

print("\n" + "=" * 80, flush=True)
print("IMPORT SUMMARY", flush=True)
print("=" * 80, flush=True)
print(f"PyTorch      : {torch.__version__}", flush=True)
print(f"CUDA OK      : {torch.cuda.is_available()}", flush=True)
N_GPUS = torch.cuda.device_count()
for i in range(N_GPUS):
    props = torch.cuda.get_device_properties(i)
    mem_gb = getattr(props, "total_memory", getattr(props, "total_mem", 0)) / 1024**3
    print(f"GPU {i}        : {props.name}  ({mem_gb:.1f} GB)", flush=True)
import vllm as _vllm_mod
import transformers as _tf_mod
print(f"vLLM version : {_vllm_mod.__version__}", flush=True)
print(f"Transformers : {_tf_mod.__version__}", flush=True)
print("All imports OK", flush=True)

# LightGBM (ablation only)
try:
    import lightgbm
except ImportError:
    _pip("lightgbm")
print("lightgbm: ready", flush=True)


STAGE 0 vLLM compatibility installer
IMPORTANT: if this cell imported torch/vllm failed before, restart kernel and run from top.
Pre-installed vLLM: FAIL (No module named 'vllm')
  Trying transformers==4.48.0 + vllm... 

2026-06-06 00:45:03.705526: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1780706703.892793      58 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1780706703.944348      58 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1780706704.394800      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780706704.394832      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780706704.394835      58 computation_placer.cc:177] computation placer alr

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

WARNING 06-06 00:45:24 [config.py:70] Support for Transformers v4 is deprecated. The Transformers v4 codepath will become unmaintained in vLLM v0.22.0 and will be removed in vLLM v0.24.0. Please upgrade to Transformers v5: pip install --upgrade transformers
OK
z3-solver: already OK

IMPORT SUMMARY
PyTorch      : 2.11.0+cu130
CUDA OK      : True
GPU 0        : Tesla T4  (14.6 GB)
GPU 1        : Tesla T4  (14.6 GB)
vLLM version : 0.22.1
Transformers : 4.57.6
All imports OK
lightgbm: ready


In [4]:
# ================= BOOT CHECK =================
# Xac nhan kernel/GPU SAU khi Stage 0 da import vLLM/torch OK.
# Neu cell nay khong in ra, Kaggle dang ket o "Adding data" -> bo input nang, chi add file .zip.
import os, sys, glob, subprocess
print("="*70, flush=True)
print("  BOOT CHECK", flush=True)
print("="*70, flush=True)
print(f"  Python : {sys.version.split()[0]}", flush=True)
try:
    import torch
    print(f"  Torch  : {torch.__version__}  CUDA available={torch.cuda.is_available()}", flush=True)
    if torch.cuda.is_available():
        print(f"  GPU    : {torch.cuda.get_device_name(0)}", flush=True)
except Exception as e:
    print("  Torch check failed:", repr(e), flush=True)
print("  /kaggle/input entries:", flush=True)
for p in sorted(glob.glob('/kaggle/input/*'))[:30]:
    print("   -", p, flush=True)
print("="*70, flush=True)


  BOOT CHECK
  Python : 3.12.13
  Torch  : 2.11.0+cu130  CUDA available=True
  GPU    : Tesla T4
  /kaggle/input entries:
   - /kaggle/input/datasets
   - /kaggle/input/models
   - /kaggle/input/notebooks


In [5]:
# ================= LOCATE + UNZIP LoRA (v16) =================
# Tim file qwen3_cot_lora_v16.zip o BAT KY dau trong /kaggle/input,
# unzip vao /kaggle/working, roi set COT_LORA_PATH. Fail som neu khong thay.
import os, glob, zipfile

_UNZIP_DEST = "/kaggle/working"
_zip_hits = glob.glob("/kaggle/input/**/qwen3_cot_lora_v16.zip", recursive=True)
print("="*70, flush=True)
print(f"  LOCATE LoRA v16", flush=True)
print("="*70, flush=True)

_resolved = None
if _zip_hits:
    zpath = _zip_hits[0]
    print(f"  Found zip: {zpath}", flush=True)
    with zipfile.ZipFile(zpath) as zf:
        zf.extractall(_UNZIP_DEST)
    # adapter dir is /kaggle/working/qwen3_cot_lora_v16
    cand = os.path.join(_UNZIP_DEST, "qwen3_cot_lora_v16")
    if os.path.exists(os.path.join(cand, "adapter_config.json")):
        _resolved = cand
    else:
        # zip might extract flat; search
        for root, _, files in os.walk(_UNZIP_DEST):
            if "adapter_config.json" in files:
                _resolved = root; break
    print(f"  Unzipped -> {_resolved}", flush=True)
else:
    print("  No zip found. Falling back to unzipped adapter dir search...", flush=True)
    for root, _, files in os.walk("/kaggle/input"):
        if "adapter_config.json" in files and "qwen3_cot_lora_v16" in root:
            _resolved = root; break

# ---- ASSERTS: fail early with clear message ----
assert _resolved is not None, (
    "COULD NOT FIND LoRA v16. Add the finetune notebook output (which contains "
    "qwen3_cot_lora_v16.zip) as Input, or upload the zip as a Dataset.")
_cfg  = os.path.join(_resolved, "adapter_config.json")
_safe = os.path.join(_resolved, "adapter_model.safetensors")
assert os.path.exists(_cfg),  f"MISSING adapter_config.json in {_resolved}"
assert os.path.exists(_safe), f"MISSING adapter_model.safetensors in {_resolved}"

COT_LORA_PATH = _resolved
print(f"\n  [OK] adapter_config.json", flush=True)
print(f"  [OK] adapter_model.safetensors ({os.path.getsize(_safe)/1e6:.1f} MB)", flush=True)
print(f"  COT_LORA_PATH = {COT_LORA_PATH}", flush=True)


  LOCATE LoRA v16
  Found zip: /kaggle/input/notebooks/selaninguyen/v16-finetune/qwen3_cot_lora_v16.zip
  Unzipped -> /kaggle/working/qwen3_cot_lora_v16

  [OK] adapter_config.json
  [OK] adapter_model.safetensors (174.7 MB)
  COT_LORA_PATH = /kaggle/working/qwen3_cot_lora_v16


In [6]:

# ---- Robust Kaggle path resolver (handles mount-path variations) ----
import os as _os
def _resolve(cands, label="path"):
    for p in cands:
        if _os.path.exists(p):
            print(f"  resolved {label}: {p}")
            return p
    print(f"  WARNING {label}: none of candidates exist; using first: {cands[0]}")
    return cands[0]

TRAIN_PATH = _resolve([
    "/kaggle/input/test-pipeline/Logic_Based_Educational_Queries_final_v4.json",
    "/kaggle/input/datasets/nguyenminhtric/test-pipeline/Logic_Based_Educational_Queries_final_v4.json",
], "TRAIN")
TEST_PATH = _resolve([
    "/kaggle/input/test-pipeline/generated_v4style_300.json",
    "/kaggle/input/datasets/nguyenminhtric/test-pipeline/generated_v4style_300.json",
], "TEST")
DATASET_PATH = TRAIN_PATH    # eval notebooks use the labeled train file


# ==================================================================
# CAU HINH -- Chinh sua o day
# ==================================================================
QWEN_MODEL_ID  = "/kaggle/input/models/qwen-lm/qwen-3/transformers/8b/1"
# DATASET_PATH set by resolver
PHYSICS = "/kaggle/input/datasets/nguyenminhtric/test-pipeline/Physics_Problems.csv"# DATASET_PATH set by resolver  # v8.2: Skip physics, focus on logic

# --- Data Split Config ---
SEED = 42
TRAIN_RATIO = 0.85
VAL_RATIO   = 0.10
TEST_RATIO  = 0.05
RUN_ON_SPLIT = "all"  # v8.2: Full dataset  # 'test', 'train', 'val', hoac 'all'

N_SAMPLES      = 411
# --- Best-of-N Config ---
BEST_OF_N       = 3  # v8.2: Reduced for time budget     # Number of candidates per sample
BON_TEMPERATURE = 0.5  # v8.2: Lower for consistency   # Higher temp for diversity

# --- v9: Optional CoT for Formalization (Pass 1) ---
# False = giu nguyen baseline v8.2 (da kiem chung 50.0%).
# True  = cho LLM "thinking" truoc khi chot JSON AST -> ky vong giam no_ast, danh thuc Z3.
FORMALIZE_WITH_COT = False

MAX_RETRIES    = 2               # Giam xuong 2 vi v4 nhe hon
OUTPUT_PATH    = "/kaggle/working/pipeline_results_v12_1.json"
MAX_NEW_TOKENS_PASS1 = 4096      # Token cho Premises
MAX_NEW_TOKENS_PASS2 = 1024      # Token cho Question (it hon)
ANS_MAX_TOKENS       = 600       # Token cho Qwen Fallback

# vLLM Config
TENSOR_PARALLEL  = min(N_GPUS, 2)
MAX_MODEL_LEN    = 8192
GPU_MEMORY_UTIL  = 0.85
DTYPE            = "half"

# Physics Config
PHYSICS_MODE       = "direct"
PHYSICS_MAX_TOKENS = 1024
PHYSICS_TOLERANCE  = 0.05

# Z3 Entailment Config
Z3_ENTAILMENT      = True
Z3_SOLVER_TIMEOUT  = 5000
Z3_SELF_CORRECT    = False   # v10: OFF (24/114 resolved, 9 -> Unknown; low ROI)
Z3_SC_MAX_RETRIES  = 1       # Max self-correction rounds
REPETITION_PENALTY = 1.1     # v8.1 from exact.ipynb: chong token loop
# ==================================================================

# ==================================================================
# v10 -- Z3 FIDELITY CONFIG
# ==================================================================
# Quality Gate: chi cho Z3 tra loi khi formalization dang tin cay.
Z3_QUALITY_GATE          = True
GATE_REQUIRE_FULL_COMPILE = True   # compiled_count == total_count
GATE_REJECT_HALLUCINATION = True   # 0 hallucination warning
# Neu gate fail -> cau hoi do chuyen sang Qwen CoT fallback.

# Formalizer-LoRA (STaR loop). De rong "" = dung model goc cho moi pass.
FORMALIZER_LORA_PATH = ""          # vd: "/kaggle/working/qwen3_formalizer_lora"
LORA_MAX_RANK        = 16

# Export verified formalizations (chay tren TRAIN de tao data train cho LoRA)
EXPORT_VERIFIED        = False
VERIFIED_OUTPUT_PATH   = "/kaggle/working/verified_formalizations.json"

# ==================================================================
# v11 -- GOLD-FOL CONFIG
# ==================================================================
PREMISE_SOURCE   = "gold"     # "gold" (parse premises-FOL) | "lora_fol" (NL->FOL via LoRA)
USE_IDX_FILTER   = False      # True: chi dung premise trong idx[q] (gold supporting set)
FOL_FALLBACK_TO_QWEN = True   # mau parse FOL that bai -> Qwen CoT truc tiep

# ==================================================================
# v12 -- BoN + SELF-CONSISTENCY + GATING + UNSAT-CORE
# ==================================================================
BON_N_QFORMALIZE     = 5           # candidates per Pass-2 question
BON_TEMP             = 0.7         # generation diversity
SC_CONFIDENCE_TAU    = 0.6         # Tier-1 vote-share threshold
GROUNDED_FRAC_TAU    = 0.6         # Tier-1 grounded-candidates threshold
EXTRACT_UNSAT_CORE   = True        # produce support_idx for Z3=Yes

# v12.1 -- post-mortem fixes
USE_Z3_INFORMED_FALLBACK = True   # Qwen fallback sees premise FOL + Z3 votes + Z3 verdict
                                  # (v12 showed qwen_fallback @ 36%; this aims for 45-50%)
MAX_PASS2_TOKENS_YN  = 200
MAX_PASS2_TOKENS_MC  = 500

# v12 overrides v11 defaults:
USE_IDX_FILTER       = True        # ON by default in v12 (was False in v11)

print("Config OK - v12.1 (Z3-informed fallback + Qwen-priority arbiter)")

# ==================================================================
# v13 -- ARCHITECTURE COMPARISON
# ==================================================================
CAL_SEED            = 42
CAL_TRAIN_RATIO     = 0.80
CAL_TAU             = 0.55         # overridden by tuned tau in v13.1/v13.3
PH_HIGH_CONF        = 0.7          # Parallel-Hybrid threshold: trust Z3 on disagreement
FEATURES_CACHE_PATH = "/kaggle/working/pipeline_features_16_1.json"
LOAD_FEATURES_CACHE = True         # use cache if exists; else build & save
FORCE_REBUILD = True        # set True to ignore cache

print("v13 config extras loaded")

# ==================================================================
# v13.4 -- LoRA v14 for Qwen-CoT step
# ==================================================================
# Path to v14 LoRA adapter (PEFT format). Upload from previous Kaggle output.
# COT_LORA_PATH is set by the LOCATE+UNZIP cell that runs before this.
# (kept out of config so inference does not depend on importing the finetune notebook)
assert "COT_LORA_PATH" in globals(), "Run the LOCATE+UNZIP LoRA cell first!"
USE_COT_LORA  = True

# Override v13 cache so we don\'t collide with base-Qwen features
FEATURES_CACHE_PATH = "/kaggle/working/pipeline_features_16_1.json"

print("v13.4 config:")
print(f"  COT_LORA_PATH       = {COT_LORA_PATH}")
print(f"  USE_COT_LORA        = {USE_COT_LORA}")
print(f"  FEATURES_CACHE_PATH = {FEATURES_CACHE_PATH}")

# ==================================================================
# v13.5 -- System prompt fix (CRITICAL: must match v14 training prompt)
# ==================================================================
# This EXACT prompt was used to train LoRA v14. Must be identical at inference.
V14_COT_SYSTEM = (
    "You are a careful logic tutor. Given a list of premises and a question, "
    "reason step-by-step by referencing specific premises (e.g. 'From premise 3...'). "
    "Then state your conclusion on a final line in the exact format: 'Final Answer: X' "
    "where X is one of: Yes, No, Unknown, A, B, C, or D."
)
COT_MAX_TOKENS = 768    # explanations avg ~85 tok; 768 gives ample room

# New cache (old v14 cache has wrong-prompt qwen answers, must rebuild)
FEATURES_CACHE_PATH = "/kaggle/working/pipeline_features_16_1.json"
FORCE_REBUILD = True    # set False after first run

print("v13.5 config: V14_COT_SYSTEM defined, cache -> pipeline_features_v13_5.json")

# ==================================================================
# v13.6 -- Option B (Qwen-CoT BoN+SC) + Option C (richer calibrator features)
# ==================================================================
QWEN_BON_N      = 5       # Qwen-CoT self-consistency samples per question
QWEN_BON_TEMP   = 0.5     # diversity for SC voting
QWEN_CONF_TRUST = 0.6     # conf_hybrid: trust Qwen on disagreement if SC-conf >= this

FEATURES_CACHE_PATH = "/kaggle/working/pipeline_features_16_1.json"
FORCE_REBUILD = True      # set False after first run

print("v13.6 config: Qwen BoN-SC + richer features, cache -> pipeline_features_v13_6.json")


  resolved TRAIN: /kaggle/input/datasets/nguyenminhtric/test-pipeline/Logic_Based_Educational_Queries_final_v4.json
  resolved TEST: /kaggle/input/datasets/nguyenminhtric/test-pipeline/generated_v4style_300.json
Config OK - v12.1 (Z3-informed fallback + Qwen-priority arbiter)
v13 config extras loaded
v13.4 config:
  COT_LORA_PATH       = /kaggle/working/qwen3_cot_lora_v16
  USE_COT_LORA        = True
  FEATURES_CACHE_PATH = /kaggle/working/pipeline_features_16_1.json
v13.5 config: V14_COT_SYSTEM defined, cache -> pipeline_features_v13_5.json
v13.6 config: Qwen BoN-SC + richer features, cache -> pipeline_features_v13_6.json


In [7]:

# ==================================================================
# STAGE 1 -- Load vLLM Engine
# ==================================================================

print(f"\nLoading vLLM engine ({QWEN_MODEL_ID})...")
print("  (Lan dau tai ~15 GB tu HuggingFace, sau do cache)")

t_load = time.time()

_USE_LORA = bool(FORMALIZER_LORA_PATH) or bool(COT_LORA_PATH and USE_COT_LORA)
llm = LLM(
    model=QWEN_MODEL_ID,
    tensor_parallel_size=TENSOR_PARALLEL,
    dtype=DTYPE,
    max_model_len=MAX_MODEL_LEN,
    gpu_memory_utilization=GPU_MEMORY_UTIL,
    trust_remote_code=True,
    enforce_eager=True,        # v8.1 FIX: Disable CUDA graph to save 1.67 GiB VRAM on T4
    enable_lora=_USE_LORA,     # v10: serve formalizer LoRA if provided
    max_lora_rank=LORA_MAX_RANK,
)

# v10: build LoRA request (used ONLY for formalization passes)
LORA_REQ = None
# v13.4 fix: guard specifically on FORMALIZER_LORA_PATH (not _USE_LORA,
# which can be True just because COT_LORA_PATH is set)
if FORMALIZER_LORA_PATH:
    from vllm.lora.request import LoRARequest
    LORA_REQ = LoRARequest("formalizer", 1, FORMALIZER_LORA_PATH)
    print(f"Formalizer-LoRA enabled: {FORMALIZER_LORA_PATH}")
else:
    print("No formalizer LoRA (base model for all passes)")

tokenizer = AutoTokenizer.from_pretrained(
    QWEN_MODEL_ID, trust_remote_code=True
)

t_loaded = time.time() - t_load
print(f"vLLM Engine loaded in {t_loaded:.1f}s")
print("OK")

# ==================================================================
# v13.4 -- Build CoT LoRA request (separate lora_int_id from formalizer)
# ==================================================================
COT_LORA_REQ = None
if COT_LORA_PATH and USE_COT_LORA:
    import os
    if os.path.exists(COT_LORA_PATH):
        from vllm.lora.request import LoRARequest
        COT_LORA_REQ = LoRARequest("cot_v14", 2, COT_LORA_PATH)
        print(f"CoT-LoRA enabled: {COT_LORA_PATH}")
    else:
        print(f"WARNING: COT_LORA_PATH not found at {COT_LORA_PATH}; falling back to base Qwen for CoT")
        USE_COT_LORA = False
else:
    print("CoT-LoRA disabled (base model for Qwen-CoT step)")



Loading vLLM engine (/kaggle/input/models/qwen-lm/qwen-3/transformers/8b/1)...
  (Lan dau tai ~15 GB tu HuggingFace, sau do cache)
WARNING 06-06 00:55:42 [envs.py:2057] Unknown vLLM environment variable detected: VLLM_USE_V1
WARNING 06-06 00:55:42 [envs.py:2057] Unknown vLLM environment variable detected: VLLM_ATTENTION_BACKEND
WARNING 06-06 00:56:03 [model.py:2090] Casting torch.bfloat16 to torch.float16.
WARNING 06-06 00:56:03 [vllm.py:1033] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
WARNING 06-06 00:56:03 [vllm.py:1058] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.


2026-06-06 00:56:13.182969: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1780707373.209522     238 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1780707373.217866     238 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1780707373.237947     238 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780707373.237996     238 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780707373.238002     238 computation_placer.cc:177] computation placer alr

WARNING 06-06 00:56:21 [config.py:70] Support for Transformers v4 is deprecated. The Transformers v4 codepath will become unmaintained in vLLM v0.22.0 and will be removed in vLLM v0.24.0. Please upgrade to Transformers v5: pip install --upgrade transformers
(EngineCore pid=238) WARNING 06-06 00:56:23 [multiproc_executor.py:1029] Reducing Torch parallelism from 2 threads to 1 to avoid unnecessary CPU contention. Set OMP_NUM_THREADS in the external environment to tune this value as needed.


2026-06-06 00:56:28.999609: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1780707389.024543     266 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1780707389.032879     266 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1780707389.052224     266 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780707389.052276     266 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780707389.052282     266 computation_placer.cc:177] computation placer alr

WARNING 06-06 00:56:37 [config.py:70] Support for Transformers v4 is deprecated. The Transformers v4 codepath will become unmaintained in vLLM v0.22.0 and will be removed in vLLM v0.24.0. Please upgrade to Transformers v5: pip install --upgrade transformers
WARNING 06-06 00:56:37 [config.py:70] Support for Transformers v4 is deprecated. The Transformers v4 codepath will become unmaintained in vLLM v0.22.0 and will be removed in vLLM v0.24.0. Please upgrade to Transformers v5: pip install --upgrade transformers
(Worker pid=266) WARNING 06-06 00:56:41 [symm_mem.py:66] SymmMemCommunicator: Device capability 7.5 not supported, communicator is not available.
(Worker pid=265) WARNING 06-06 00:56:41 [symm_mem.py:66] SymmMemCommunicator: Device capability 7.5 not supported, communicator is not available.
(Worker_TP0 pid=265) ERROR 06-06 00:56:42 [fa_utils.py:171] Cannot use FA version 2 is not supported due to FA2 is only supported on devices with compute capability >= 8
(Worker_TP1 pid=266) E

Loading safetensors checkpoint shards:   0% Completed | 0/5 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  20% Completed | 1/5 [00:19<01:19, 19.80s/it]
Loading safetensors checkpoint shards:  40% Completed | 2/5 [00:24<00:32, 10.96s/it]
Loading safetensors checkpoint shards:  60% Completed | 3/5 [00:27<00:14,  7.47s/it]
Loading safetensors checkpoint shards:  80% Completed | 4/5 [00:30<00:05,  5.50s/it]
Loading safetensors checkpoint shards: 100% Completed | 5/5 [00:31<00:00,  3.95s/it]
Loading safetensors checkpoint shards: 100% Completed | 5/5 [00:31<00:00,  6.31s/it]
(Worker_TP0 pid=265) 


(Worker_TP0 pid=265) WARNING 06-06 00:57:22 [utils.py:279] Using default LoRA kernel configs
(EngineCore pid=238) WARNING 06-06 00:58:03 [vllm.py:1033] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
(EngineCore pid=238) WARNING 06-06 00:58:03 [vllm.py:1058] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.
No formalizer LoRA (base model for all passes)
vLLM Engine loaded in 141.7s
OK
CoT-LoRA enabled: /kaggle/working/qwen3_cot_lora_v16


In [8]:

# ==================================================================
# STAGE 2 -- Ontology & Prompts (Two-Pass)
# ==================================================================

GLOBAL_ONTOLOGY_TEXT = """
## GLOBAL ONTOLOGY -- BAT BUOC TUAN THU

### Quantifiers:
  forall -> forall  |  exists -> exists

### Logical Operators:
  and, or, implies, iff, not

### AST Node Types (4 loai):
  quantifier : { "type":"quantifier",  "operator":"forall|exists",
                 "bound_variables":["x",...], "body":{...} }
  connective : { "type":"connective",  "operator":"and|or|implies|iff|not",
                 "operands":[{...},...] }
  predicate  : { "type":"predicate",   "name":"PredicateName",
                 "arguments":["x","y",...] }
  variable   : { "type":"variable",    "name":"x" }
  constant   : { "type":"constant",    "name":"SomeName" }

### QUY TAC:
  1. Chi dung 4 node types tren
  2. 'not' chi co DUNG 1 operand
  3. 'implies' co DUNG 2 operands
  4. bound_variables phai la list
  5. Variables: lowercase (x,y,z), Constants: PascalCase
"""

# ------------------------------------------------------------------
# FEW-SHOT EXAMPLES (CRITICAL FOR QWEN-7B)
# ------------------------------------------------------------------

FEW_SHOT_PREMISES = """
### VÍ DỤ HOÀN CHỈNH:

Premises:
  "All students who study hard pass the exam."
  "Alice is a student."
  "Alice studies hard."

Output:
{
  "step1_local_ontology": [
    {"predicate": "Student",   "arity": 1, "description": "x is a student"},
    {"predicate": "StudyHard", "arity": 1, "description": "x studies hard"},
    {"predicate": "PassExam",  "arity": 1, "description": "x passes the exam"}
  ],
  "step2_premises_ast": [
    {
      "premise_id": 0,
      "source_nl": "All students who study hard pass the exam.",
      "ast": {
        "type": "quantifier", "operator": "forall",
        "bound_variables": ["x"],
        "body": {
          "type": "connective", "operator": "implies",
          "operands": [
            {"type": "connective", "operator": "and", "operands": [
              {"type": "predicate", "name": "Student",   "arguments": ["x"]},
              {"type": "predicate", "name": "StudyHard", "arguments": ["x"]}
            ]},
            {"type": "predicate", "name": "PassExam", "arguments": ["x"]}
          ]
        }
      }
    },
    { "premise_id": 1, "source_nl": "Alice is a student.",
      "ast": {"type": "predicate", "name": "Student", "arguments": ["Alice"]} },
    { "premise_id": 2, "source_nl": "Alice studies hard.",
      "ast": {"type": "predicate", "name": "StudyHard", "arguments": ["Alice"]} }
  ]
}
"""

FEW_SHOT_QUESTIONS = """
### VÍ DỤ HOÀN CHỈNH:
Question 1: "Is it true that Alice passes the exam?" (Yes/No)
Question 2: "Which is correct? A. Alice fails. B. Alice passes." (Multiple Choice)

Output:
{
  "step3_question_fol": [
    {
      "question_id": 0,
      "question_type": "yes_no",
      "statement_ast": {"type": "predicate", "name": "PassExam", "arguments": ["Alice"]}
    },
    {
      "question_id": 1,
      "question_type": "multiple_choice",
      "options_ast": {
         "A": {"type": "connective", "operator": "not", "operands": [{"type": "predicate", "name": "PassExam", "arguments": ["Alice"]}]},
         "B": {"type": "predicate", "name": "PassExam", "arguments": ["Alice"]}
      }
    }
  ]
}
"""

# ------------------------------------------------------------------
# PASS 1 PROMPTS (PREMISES ONLY)
# ------------------------------------------------------------------

PREMISE_FORMALIZATION_SYSTEM = (
    "Ban la chuyen gia logic hinh thuc (FOL). Nhiem vu:\n"
    "  Buoc 1: Tao LOCAL ONTOLOGY -- danh sach Predicates\n"
    "  Buoc 2: Chuyen TUNG tien de thanh cay AST JSON de quy\n\n"
    + GLOBAL_ONTOLOGY_TEXT + "\n"
    + FEW_SHOT_PREMISES + "\n"
    "Output JSON THUAN TUY (khong markdown, khong text thua):\n"
    '{\n'
    '  "step1_local_ontology": [\n'
    '    {"predicate": "Name", "arity": N, "description": "..."}\n'
    '  ],\n'
    '  "step2_premises_ast": [\n'
    '    {"premise_id": 0, "source_nl": "...", "ast": { <AST> }}\n'
    '  ]\n'
    '}\n'
)

CORRECTION_SYSTEM = (
    "Ban la chuyen gia sua loi FOL. He thong Z3 da phat hien loi.\n"
    "Nhiem vu: sua lai Buoc 1 + Buoc 2 de khong con loi.\n\n"
    + GLOBAL_ONTOLOGY_TEXT + "\n"
    "Output JSON thuan tuy -- format GIONG HET lan dau.\n"
)

# ------------------------------------------------------------------
# PASS 2 PROMPTS (QUESTIONS ONLY)
# ------------------------------------------------------------------

QUESTION_FORMALIZATION_SYSTEM = (
    "Ban la chuyen gia logic hinh thuc (FOL). Nhiem vu:\n"
    "  Chuyen TUNG cau hoi thanh AST JSON de kiem tra Z3 Entailment.\n\n"
    + GLOBAL_ONTOLOGY_TEXT + "\n"
    + FEW_SHOT_QUESTIONS + "\n"
    "QUAN TRONG:\n"
    "  - Ban PHAI su dung cac Predicate tu LOCAL ONTOLOGY duoc cung cap.\n"
    "  - question_type: 'yes_no' hoac 'multiple_choice'\n"
    "  - yes_no: chuyen statement thanh 1 AST node (statement_ast)\n"
    "  - multiple_choice: chuyen TUNG option A/B/C/D thanh AST (options_ast)\n\n"
    "Output JSON THUAN TUY:\n"
    '{\n'
    '  "step3_question_fol": [\n'
    '    {\n'
    '      "question_id": 0,\n'
    '      "question_type": "yes_no",\n'
    '      "source_nl": "statement to check",\n'
    '      "statement_ast": { <AST> }\n'
    '    }\n'
    '  ]\n'
    '}\n'
)

# Fallback
# Fallback -- exact.ipynb CoT format
ANSWER_SYSTEM = (
    "You are a rigorous logical reasoning assistant specializing in First-Order Logic (FOL).\n"
    "Given a set of premises (in natural language and FOL notation), you must:\n"
    "1. Analyze the logical structure of each premise carefully.\n"
    "2. Apply formal inference rules: modus ponens, contrapositive, universal instantiation.\n"
    "3. Reason step-by-step (Chain of Thought) BEFORE committing to a final answer.\n"
    "4. For multiple-choice questions (A/B/C/D): evaluate each option against the premises.\n"
    "5. For Yes/No questions: verify the statement logically.\n"
    "6. If the premises are INSUFFICIENT, your Final Answer MUST be exactly: Unknown\n"
    "7. Never hallucinate a conclusion not entailed by the premises.\n"
    "Format your response EXACTLY as:\n"
    "### Step-by-Step Reasoning\n"
    "<your reasoning here>\n"
    "### Final Answer\n"
    "<A or B or C or D or Yes or No or Unknown>"
)

# Physics
PHYSICS_SOLVER_SYSTEM = (
    "You are an expert physics problem solver.\n"
    "Solve the problem step-by-step, showing all calculations.\n"
    "Format your response EXACTLY as:\n"
    "### Step-by-Step Reasoning\n"
    "<your detailed solution here>\n"
    "### Final Answer\n"
    "<your numerical answer with unit>"
)
PHYSICS_MC_SYSTEM = (
    "You are an expert physics problem solver.\n"
    "Solve the multiple-choice problem step-by-step.\n"
    "Evaluate each option against the physics principles.\n"
    "Format your response EXACTLY as:\n"
    "### Step-by-Step Reasoning\n"
    "<your detailed solution here>\n"
    "### Final Answer\n"
    "<A or B or C or D or your numerical answer>"
)

# ==================================================================
# UTILS
# ==================================================================
import os, csv, re, time
from collections import Counter
from dataclasses import dataclass, field
from pathlib import Path

def load_dataset(path, is_physics=False, max_samples=N_SAMPLES, split_mode=RUN_ON_SPLIT):
    if not path or not os.path.exists(path): return []
    if path.endswith(".csv"):
        with open(path, encoding="utf-8") as f: raw = list(csv.DictReader(f))
    else:
        with open(path, encoding="utf-8") as f: raw = json.load(f)
    
    out = raw[:max_samples]
    
    # --- SPLIT LOGIC ---
    import random
    rng = random.Random(SEED)
    rng.shuffle(out)
    n = len(out)
    n_train = int(n * TRAIN_RATIO)
    n_val   = int(n * VAL_RATIO)
    
    if split_mode == "train":
        out = out[:n_train]
    elif split_mode == "val":
        out = out[n_train:n_train+n_val]
    elif split_mode == "test":
        out = out[n_train+n_val:]
    # if split_mode == "all", keep out as is

    if is_physics and out:
        for s in out:
            s["premises-NL"] = []
            s["questions"]   = [s.get("question", "")]
            s["answers"]     = [str(s.get("answer", "Unknown"))]
            s["_unit"]       = s.get("unit", "")
            s["_is_physics"] = True
    return out

def safe_json(text):
    text = text.strip()
    # v7 FIX: Strip Qwen3 <think>...</think> tags
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()
    try: return json.loads(text)
    except: pass
    m = re.search(r"```(?:json)?\s*\n?(.*?)```", text, re.DOTALL)
    if m:
        try: return json.loads(m.group(1).strip())
        except: pass
    start = text.find("{")
    if start >= 0:
        depth, end = 0, start
        for i in range(start, len(text)):
            if text[i] == "{": depth += 1
            elif text[i] == "}":
                depth -= 1
                if depth == 0:
                    end = i
                    break
        try: return json.loads(text[start : end + 1])
        except: pass
    return {}

def batch_generate(prompt_pairs, max_tokens, enable_thinking=True, use_lora=False):
    formatted = []
    for sys_msg, usr_msg in prompt_pairs:
        messages = [{"role": "system", "content": sys_msg}, {"role": "user", "content": usr_msg}]
        try:
            formatted.append(tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=enable_thinking))
        except TypeError:
            formatted.append(tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))
    params = SamplingParams(temperature=0.05, max_tokens=max_tokens, repetition_penalty=1.1)
    _lr = LORA_REQ if (use_lora and LORA_REQ is not None) else None
    outputs = llm.generate(formatted, params, lora_request=_lr) if _lr else llm.generate(formatted, params)
    outputs_sorted = sorted(outputs, key=lambda x: int(x.request_id))
    return [o.outputs[0].text.strip() for o in outputs_sorted]

def batch_generate_bon(prompt_pairs, max_tokens, n=BEST_OF_N, enable_thinking=True, use_lora=False):
    """Generate N candidates per prompt using higher temperature."""
    formatted = []
    for sys_msg, usr_msg in prompt_pairs:
        messages = [{"role": "system", "content": sys_msg},
                    {"role": "user", "content": usr_msg}]
        try:
            formatted.append(tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True, enable_thinking=enable_thinking))
        except TypeError:
            formatted.append(tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True))

    params = SamplingParams(
        temperature=BON_TEMPERATURE,
        max_tokens=max_tokens,
        repetition_penalty=1.1,
        n=n,
    )
    _lr = LORA_REQ if (use_lora and LORA_REQ is not None) else None
    outputs = llm.generate(formatted, params, lora_request=_lr) if _lr else llm.generate(formatted, params)
    outputs_sorted = sorted(outputs, key=lambda x: int(x.request_id))

    return [
        [o.text.strip() for o in out.outputs]
        for out in outputs_sorted
    ]


def z3_select_best(candidates):
    """
    v8.1: Returns (data, status, func_cache, const_map).
    """
    PRIORITY = {"sat": 4, "unsat": 3, "unknown": 2, "compile_error": 1, "no_ast": 0}
    best_data, best_status, best_score = {}, "no_ast", -1
    best_func_cache = {}
    best_const_map = {}

    for raw in candidates:
        data = safe_json(raw)
        premises_ast = data.get("step2_premises_ast", [])

        if not premises_ast:
            score = PRIORITY["no_ast"]
            status = "no_ast"
            candidate_cache = {}
            candidate_const = {}
        else:
            candidate_cache = {}
            candidate_const = {}
            z3_info = verify_with_z3(premises_ast, func_cache=candidate_cache, const_map=candidate_const)
            status = z3_info["status"]
            score = PRIORITY.get(status, 0)

        if score > best_score:
            best_score = score
            best_status = status
            best_data = data
            best_func_cache = candidate_cache
            best_const_map = candidate_const

        if best_score >= 4:  # sat -> stop early
            break

    return best_data, best_status, best_func_cache, best_const_map




import re

def extract_final_answer(model_output):
    """
    v8.1: Robust answer extraction from exact.ipynb
    Multi-pattern fallback to minimize UNPARSEABLE results.
    Priority: Final Answer block > inline patterns > standalone letters > keyword scan
    """
    text = model_output.strip()

    # Pattern 1: "### Final Answer" block
    match = re.search(
        r"###\s*Final\s*Answer[:\s]*\n?\s*(.+)",
        text, re.IGNORECASE
    )
    if match:
        ans = match.group(1).strip().split("\n")[0].strip()
        if re.match(r"^unknown", ans, re.IGNORECASE):
            return "Unknown"
        m_letter = re.match(r"^([A-D])[.)\s:]", ans, re.IGNORECASE)
        if m_letter:
            return m_letter.group(1).upper()
        if re.match(r"^[A-D]$", ans, re.IGNORECASE):
            return ans.upper()
        if re.match(r"^yes", ans, re.IGNORECASE):
            return "Yes"
        if re.match(r"^no\b", ans, re.IGNORECASE):
            return "No"

    # Pattern 2: "answer is X" or "Answer: X"
    match2 = re.search(
        r"(?:answer\s*(?:is|:)\s*)([A-D]|unknown|yes|no)",
        text, re.IGNORECASE
    )
    if match2:
        g = match2.group(1).strip()
        if g.lower() == "unknown": return "Unknown"
        if g.lower() == "yes":    return "Yes"
        if g.lower() == "no":     return "No"
        return g.upper()

    # Pattern 3: Standalone letter near end of text
    last_500 = text[-500:]
    match3 = re.findall(r"\b([A-D])\b", last_500)
    if match3:
        return match3[-1].upper()

    # Pattern 4: Unknown/Yes/No anywhere
    if re.search(r"\bunknown\b", text, re.IGNORECASE):
        return "Unknown"
    if re.search(r"\byes\b", text, re.IGNORECASE):
        return "Yes"
    if re.search(r"\bno\b", text, re.IGNORECASE):
        return "No"

    return "UNPARSEABLE"

print("extract_final_answer v8.1 (from exact.ipynb) san sang")
def extract_physics_answer(model_output):
    """Extract answer from physics CoT response (v8.2)."""
    text = model_output.strip()
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()
    # Pattern 1: ### Final Answer block
    match = re.search(r"###\s*Final\s*Answer[:\s]*\n?\s*(.+)", text, re.IGNORECASE)
    if match:
        ans = match.group(1).strip().split("\n")[0].strip()
        ans = re.sub(r"^(?:the\s+answer\s+is|answer\s*[:=])\s*", "", ans, flags=re.IGNORECASE).strip()
        m_letter = re.match(r"^([A-D])[.)\s:]", ans, re.IGNORECASE)
        if m_letter:
            return m_letter.group(1).upper()
        if re.match(r"^[A-D]$", ans, re.IGNORECASE):
            return ans.upper()
        return ans
    # Pattern 2: JSON fallback
    data = safe_json(text)
    if data.get("answer"):
        return str(data["answer"]).strip()
    # Pattern 3: "answer is X"
    match2 = re.search(r"answer\s*(?:is|:|=)\s*(.+?)(?:\n|$)", text, re.IGNORECASE)
    if match2:
        return match2.group(1).strip()
    return "Unknown"

print("extract_physics_answer v8.2 san sang")



extract_final_answer v8.1 (from exact.ipynb) san sang
extract_physics_answer v8.2 san sang


In [9]:

# ==================================================================
# STAGE 3 -- Z3 Compiler + Entailment Checker (v8.1 HARDENED)
# ==================================================================
# v8.1 over v7:
#   1. const_map: Deterministic constant->IntVal mapping (no hash collision)
#   2. SAFE fuzzy: case-insensitive + prefix-strip ONLY (no substring!)
#   3. MC elimination: if 3/4 options contradicted, pick the survivor
# v8.1 over v8:
#   - REMOVED dangerous substring matching
#   - Self-Correction will NOT override Qwen fallback with Unknown
# ==================================================================


def get_z3_func(name, arity, func_cache):
    """Get or create Z3 Function with SAFE fuzzy matching.
    v8.1: Only case-insensitive + prefix-strip. NO substring matching."""
    key = f"{name}_{arity}"
    if key in func_cache:
        return func_cache[key]

    # --- v8.1: SAFE fuzzy match (no substring!) ---
    def _normalize(n):
        """Strip Is/Has/Can prefix for matching."""
        n_low = n.lower()
        # v10 FIX: removed can/can_ -- modal verbs (ability/permission) are NOT
        # the same predicate as their bare form (actuality). Caused wrong entailments.
        for prefix in ("is_", "has_", "is", "has"):
            if n_low.startswith(prefix) and len(n_low) > len(prefix):
                return n_low[len(prefix):]
        return n_low

    for cached_key in list(func_cache.keys()):
        parts = cached_key.rsplit("_", 1)
        if len(parts) != 2:
            continue
        cached_name, cached_arity_str = parts
        try:
            cached_arity = int(cached_arity_str)
        except ValueError:
            continue
        if cached_arity != arity:
            continue

        # Level 1: Case-insensitive exact match
        if cached_name.lower() == name.lower():
            func_cache[key] = func_cache[cached_key]
            print(f"    [FUZZY] {name}/{arity} -> {cached_name} (case)")
            return func_cache[key]

        # Level 2: Prefix-strip match (IsStudent -> student == student)
        if _normalize(name) == _normalize(cached_name):
            func_cache[key] = func_cache[cached_key]
            print(f"    [FUZZY] {name}/{arity} -> {cached_name} (prefix-strip)")
            return func_cache[key]

        # v8.1: NO Level 3 substring matching -- it caused false positives!

    # No fuzzy match -- create new function
    sorts = [IntSort()] * arity + [BoolSort()]
    func_cache[key] = Function(name, *sorts)
    return func_cache[key]


def _resolve_bound_var_name(bv):
    if isinstance(bv, dict):
        return bv.get("name", str(bv))
    return str(bv)


def _get_constant_val(name, const_map):
    """v8: Deterministic constant mapping. No hash collisions."""
    if name not in const_map:
        const_map[name] = IntVal(len(const_map) + 1)
    return const_map[name]


def _resolve_predicate_arg(a, var_map, func_cache, const_map):
    if isinstance(a, str):
        if a in var_map:
            return var_map[a]
        return _get_constant_val(a, const_map)
    if isinstance(a, dict):
        atype = a.get("type", "")
        name = a.get("name", "")
        if atype == "variable":
            if name in var_map:
                return var_map[name]
            v = Int(name)
            var_map[name] = v
            return v
        if atype == "constant":
            return _get_constant_val(name, const_map)
        raise ValueError(f"Argument khong hop le (type={atype!r})")
    return _get_constant_val(str(a), const_map)


def compile_ast(node, var_map, func_cache, const_map):
    """Compile 1 AST node -> Z3 expression.
    v8.1: Uses shared func_cache + const_map."""
    if not isinstance(node, dict):
        raise ValueError(f"Expected dict, got {type(node)}: {node!r}")

    ntype = node.get("type", "")

    if ntype == "quantifier":
        op = node.get("operator", "").lower()
        bvs = node.get("bound_variables", [])
        if not bvs:
            raise ValueError("quantifier thieu bound_variables")
        bv_names = [_resolve_bound_var_name(bv) for bv in bvs]
        z3_bvs = [Int(v) for v in bv_names]
        child_map = {**var_map, **{v: z3_bvs[i] for i, v in enumerate(bv_names)}}
        body = compile_ast(node["body"], child_map, func_cache, const_map)
        if op == "forall":
            return ForAll(z3_bvs, body)
        elif op in ("exists", "exist"):
            return Exists(z3_bvs, body)
        else:
            raise ValueError(f"Quantifier khong hop le: {op!r}")

    elif ntype == "connective":
        op = node.get("operator", "").lower()
        ops = [compile_ast(o, var_map, func_cache, const_map) for o in node.get("operands", [])]
        if op == "and":
            return And(*ops)
        elif op == "or":
            return Or(*ops)
        elif op == "implies":
            if len(ops) != 2:
                raise ValueError(f"implies can 2 operands, nhan {len(ops)}")
            return Implies(ops[0], ops[1])
        elif op == "iff":
            if len(ops) != 2:
                raise ValueError(f"iff can 2 operands, nhan {len(ops)}")
            return And(Implies(ops[0], ops[1]), Implies(ops[1], ops[0]))
        elif op == "not":
            if len(ops) != 1:
                raise ValueError(f"not can 1 operand, nhan {len(ops)}")
            return Not(ops[0])
        else:
            raise ValueError(f"Connective khong hop le: {op!r}")

    elif ntype == "predicate":
        name = node.get("name", "")
        args = node.get("arguments", [])
        if not name:
            raise ValueError('predicate thieu "name"')
        func = get_z3_func(name, len(args), func_cache)
        z3_args = [_resolve_predicate_arg(a, var_map, func_cache, const_map) for a in args]
        return func(*z3_args)

    elif ntype in ("variable", "constant"):
        name = node.get("name", "")
        if name in var_map:
            return var_map[name]
        if ntype == "constant":
            return _get_constant_val(name, const_map)
        v = Int(name)
        var_map[name] = v
        return v

    else:
        raise ValueError(f"AST node type khong hop le: {ntype!r}")


def verify_with_z3(premises_ast, func_cache=None, const_map=None):
    """Compile all premises -> Z3, check consistency."""
    if func_cache is None:
        func_cache = {}
    if const_map is None:
        const_map = {}
    solver = Solver()
    errors = []
    compiled = 0

    for item in premises_ast:
        pid = item.get("premise_id", "?")
        try:
            ast = item.get("ast", {})
            if not ast:
                errors.append(f"Premise {pid}: AST rong")
                continue
            expr = compile_ast(ast, {}, func_cache, const_map)
            solver.add(expr)
            compiled += 1
        except Exception as e:
            errors.append(f"Premise {pid}: {str(e)[:250]}")

    if errors:
        return {"status": "compile_error", "errors": errors,
                "compiled_count": compiled, "total_count": len(premises_ast)}
    try:
        result = solver.check()
        return {"status": str(result), "errors": [],
                "compiled_count": compiled, "total_count": len(premises_ast)}
    except Exception as e:
        return {"status": "solver_error", "errors": [str(e)],
                "compiled_count": compiled, "total_count": len(premises_ast)}


def hallucination_check(local_ontology, premises_ast):
    """Check: all predicates in AST must be in Local Ontology."""
    declared = set()
    for item in local_ontology:
        if isinstance(item, dict):
            declared.add(item.get("predicate", ""))

    warnings = []

    def collect_predicates(node, found):
        if not isinstance(node, dict):
            return
        if node.get("type") == "predicate":
            found.add(node.get("name", ""))
        for v in node.values():
            if isinstance(v, dict):
                collect_predicates(v, found)
            elif isinstance(v, list):
                for sub in v:
                    collect_predicates(sub, found)

    for item in premises_ast:
        used = set()
        collect_predicates(item.get("ast", {}), used)
        hallucinated = used - declared - {""}
        if hallucinated:
            warnings.append(
                f"Premise {item.get('premise_id', '?')}: "
                f"predicates not in Ontology -> {hallucinated}"
            )
    return warnings


# ==================================================================
# Z3 ENTAILMENT CHECKER (v8.1 HARDENED)
# ==================================================================

def _compile_premises_to_solver_shared(premises_ast, func_cache, const_map):
    """Compile premises using SHARED func_cache + const_map."""
    solver = Solver()
    solver.set("timeout", Z3_SOLVER_TIMEOUT)
    errors = []
    compiled = 0

    for item in premises_ast:
        pid = item.get("premise_id", "?")
        try:
            ast_node = item.get("ast", {})
            if not ast_node:
                continue
            expr = compile_ast(ast_node, {}, func_cache, const_map)
            solver.add(expr)
            compiled += 1
        except Exception as e:
            errors.append(f"P{pid}: {str(e)[:200]}")

    return solver, compiled, errors


def z3_entailment_check(premises_ast, question_fol_item, func_cache=None, const_map=None):
    """v8.1: Shared func_cache + const_map + safe fuzzy matching."""
    if func_cache is None:
        func_cache = {}
    if const_map is None:
        const_map = {}

    q_type = question_fol_item.get("question_type", "yes_no")

    if q_type == "yes_no":
        return _entail_yes_no(premises_ast, question_fol_item, func_cache, const_map)
    elif q_type == "multiple_choice":
        return _entail_mc(premises_ast, question_fol_item, func_cache, const_map)
    else:
        return {"answer": "Unknown", "method": "unsupported_qtype"}


def _entail_yes_no(premises_ast, q_item, func_cache, const_map):
    """Entailment for Yes/No questions."""
    stmt_ast = q_item.get("statement_ast", {})
    if not stmt_ast:
        return {"answer": "Unknown", "method": "no_statement_ast"}

    try:
        stmt_expr = compile_ast(stmt_ast, {}, func_cache, const_map)
    except Exception as e:
        return {"answer": "Unknown", "method": "stmt_compile_error",
                "detail": str(e)[:200]}

    # Test YES: premises + NOT(stmt) -> UNSAT?
    solver1, c1, e1 = _compile_premises_to_solver_shared(premises_ast, func_cache, const_map)
    if e1:
        return {"answer": "Unknown", "method": "premise_compile_error",
                "detail": "; ".join(e1[:3])}

    try:
        solver1.push()
        solver1.add(Not(stmt_expr))
        r1 = solver1.check()
        solver1.pop()
    except Exception as e:
        r1 = None

    if r1 == unsat:
        return {"answer": "Yes", "method": "z3_entailment",
                "detail": "premises + NOT(Q) is UNSAT => Q is entailed"}

    # Test NO: premises + stmt -> UNSAT?
    solver2, _, _ = _compile_premises_to_solver_shared(premises_ast, func_cache, const_map)
    try:
        solver2.push()
        solver2.add(stmt_expr)
        r2 = solver2.check()
        solver2.pop()
    except Exception as e:
        r2 = None

    if r2 == unsat:
        return {"answer": "No", "method": "z3_negation",
                "detail": "premises + Q is UNSAT => Q contradicts premises"}

    return {"answer": "Unknown", "method": "z3_undetermined",
            "detail": f"Neither entailed nor contradicted (r1={r1}, r2={r2})"}


def _entail_mc(premises_ast, q_item, func_cache, const_map):
    """Entailment for Multiple Choice (A/B/C/D) with elimination logic."""
    options_ast = q_item.get("options_ast", {})
    if not options_ast:
        return {"answer": "Unknown", "method": "no_options_ast"}

    entailed = []
    consistent = []
    contradicted = []
    details = {}

    for label in ("A", "B", "C", "D"):
        opt_ast = options_ast.get(label, {})
        if not opt_ast:
            continue

        try:
            opt_expr = compile_ast(opt_ast, {}, func_cache, const_map)
        except Exception as e:
            details[label] = f"compile_error: {str(e)[:100]}"
            continue

        solver, c, e = _compile_premises_to_solver_shared(premises_ast, func_cache, const_map)
        if e:
            details[label] = "premise_error"
            continue

        try:
            # Entailment: premises + NOT(option) -> UNSAT?
            solver.push()
            solver.add(Not(opt_expr))
            r = solver.check()
            solver.pop()

            if r == unsat:
                entailed.append(label)
                details[label] = "entailed"

            # Contradiction: premises + option -> UNSAT?
            solver.push()
            solver.add(opt_expr)
            r2 = solver.check()
            solver.pop()

            if r2 == unsat:
                contradicted.append(label)
                if label not in details:
                    details[label] = "contradicted"
            elif r2 == sat:
                consistent.append(label)
                if label not in details:
                    details[label] = "consistent"
        except Exception as e:
            details[label] = f"solver_error: {str(e)[:100]}"

    # Decision logic (v8 enhanced)
    if len(entailed) == 1:
        return {"answer": entailed[0], "method": "z3_unique_entailment",
                "detail": f"Only {entailed[0]} entailed. {details}"}
    elif len(entailed) > 1:
        return {"answer": entailed[0], "method": "z3_multi_entailment",
                "detail": f"Multiple entailed: {entailed}. {details}"}

    # v8: If only 1 option is NOT contradicted, pick it
    non_contradicted = [l for l in ("A", "B", "C", "D")
                        if l in consistent and l not in contradicted]
    if len(non_contradicted) == 1:
        return {"answer": non_contradicted[0], "method": "z3_elimination",
                "detail": f"Elimination: only {non_contradicted[0]} survives. {details}"}

    if len(consistent) == 1:
        return {"answer": consistent[0], "method": "z3_unique_consistent",
                "detail": f"Only {consistent[0]} consistent. {details}"}
    else:
        return {"answer": "Unknown", "method": "z3_mc_undetermined",
                "detail": f"Entailed={entailed}, Consistent={consistent}, "
                          f"Contradicted={contradicted}. {details}"}


print("Z3 compiler v8.1 (const_map + SAFE fuzzy + elimination) san sang")


Z3 compiler v8.1 (const_map + SAFE fuzzy + elimination) san sang


In [10]:
# ==================================================================
# STAGE 2.5 -- Deterministic FOL-string -> Z3 Parser (v11)
# 99.6% formula / 98.8% sample coverage on the challenge dataset.
# ==================================================================
import re
from z3 import (Int, IntSort, BoolSort, Function, ForAll, Exists, IntVal,
                And, Or, Not, Implies, Solver, sat, unsat)

TOKEN_RE = re.compile(r"∀|∃|→|↔|¬|∧|∨|≥|≤|≠|>=|<=|!=|=|>|<|\+|\-|\*|/|\(|\)|,|'[^']*'|\d+\.?\d*|[A-Za-z_][A-Za-z0-9_]*|\S")
CMP = {'=','>','<','≥','≤','≠','>=','<=','!='}
def tokenize(s):
    return TOKEN_RE.findall(s)

class FOLParser:
    """v11 parser: FOL subset + arithmetic comparisons -> Z3.
    Bool functions (predicates) and Int functions (numeric terms) kept in
    separate caches keyed by name_arity."""
    def __init__(self, func_cache, const_map, int_cache=None):
        self.func_cache = func_cache      # Bool-valued predicates
        self.const_map = const_map
        self.int_cache = int_cache if int_cache is not None else {}  # Int-valued funcs

    def parse(self, s):
        self.toks = tokenize(s); self.pos = 0
        self.scope = {}; self.free = {}
        expr = self._iff()
        if self.pos != len(self.toks):
            raise ValueError(f"trailing tokens: {self.toks[self.pos:]}")
        if self.free:
            expr = ForAll(list(self.free.values()), expr)
        return expr

    def _peek(self): return self.toks[self.pos] if self.pos < len(self.toks) else None
    def _eat(self, t=None):
        cur = self._peek()
        if cur is None: raise ValueError("unexpected end")
        if t is not None and cur != t: raise ValueError(f"expected {t}, got {cur}")
        self.pos += 1; return cur

    def _iff(self):
        left = self._implies()
        while self._peek() == '↔':
            self._eat(); right = self._implies()
            left = And(Implies(left, right), Implies(right, left))
        return left
    def _implies(self):
        left = self._or()
        if self._peek() == '→':
            self._eat(); return Implies(left, self._implies())
        return left
    def _or(self):
        left = self._and()
        while self._peek() == '∨':
            self._eat(); left = Or(left, self._and())
        return left
    def _and(self):
        left = self._not()
        while self._peek() == '∧':
            self._eat(); left = And(left, self._not())
        return left
    def _not(self):
        if self._peek() == '¬':
            self._eat(); return Not(self._not())
        return self._quant_or_atom()
    def _quant_or_atom(self):
        t = self._peek()
        if t in ('∀','∃'):
            self._eat(); var = self._eat()
            v = Int(var); saved = self.scope.get(var); self.scope[var] = v
            body = self._not()
            if saved is None: del self.scope[var]
            else: self.scope[var] = saved
            return ForAll([v], body) if t=='∀' else Exists([v], body)
        if t == '(':
            self._eat('('); e = self._iff(); self._eat(')'); return e
        return self._atom()
    def _atom(self):
        # Could be a Bool predicate, or an arithmetic comparison.
        # Try to parse an arithmetic term first; if followed by CMP -> comparison.
        start = self.pos
        term = self._arith()
        if self._peek() in CMP:
            op = self._eat(); rhs = self._arith()
            return self._cmp(op, term, rhs)
        # not a comparison: term must itself be a Bool predicate
        if term is None:
            raise ValueError("empty atom")
        return term  # _arith returns Bool pred when it was a bare predicate

    def _cmp(self, op, a, b):
        if op in ('=',): return a == b
        if op in ('≠','!='): return a != b
        if op in ('>',): return a > b
        if op in ('<',): return a < b
        if op in ('≥','>='): return a >= b
        if op in ('≤','<='): return a <= b
        raise ValueError(f"bad cmp {op}")

    def _arith(self):
        left = self._arith_term()
        while self._peek() in ('+','-'):
            op=self._eat(); right=self._arith_term()
            left = (left+right) if op=='+' else (left-right)
        return left
    def _arith_term(self):
        left = self._factor()
        while self._peek() in ('*','/'):
            op=self._eat(); right=self._factor()
            left = (left*right) if op=='*' else (left/right)
        return left
    def _factor(self):
        tok = self._peek()
        if tok == '(':
            self._eat('('); e=self._iff(); self._eat(')'); return e
        if re.match(r'^\d', tok):
            self._eat()
            return IntVal(int(float(tok)))
        if tok.startswith("'") and tok.endswith("'"):
            self._eat()
            cname = tok.strip("'")
            if cname not in self.const_map:
                self.const_map[cname] = IntVal(len(self.const_map)+1)
            return self.const_map[cname]
        # name: predicate (Bool) or numeric function (Int) depending on following CMP/arith
        name = self._eat()
        args = []
        is_call = False
        if self._peek() == '(':
            is_call=True; self._eat('(')
            if self._peek() != ')':
                args.append(self._arith())
                while self._peek()==',':
                    self._eat(','); args.append(self._arith())
            self._eat(')')
        # Decide Bool vs Int by lookahead: if next is CMP/arith-op, this name is Int-valued
        nxt = self._peek()
        numeric_ctx = nxt in CMP or nxt in ('+','-','*','/')
        if name in self.scope: return self.scope[name]
        if not is_call and re.match(r'^[a-z][a-z0-9_]*$', name) and not args:
            # bare lowercase -> free var (numeric or term)
            if name not in self.free: self.free[name]=Int(name)
            return self.free[name]
        key=f"{name}_{len(args)}"
        if numeric_ctx:
            if key not in self.int_cache:
                sorts=[IntSort()]*len(args)+[IntSort()]
                self.int_cache[key]=Function(name+"_I", *sorts)
            return self.int_cache[key](*args) if args else self.int_cache[key]()
        else:
            if key not in self.func_cache:
                sorts=[IntSort()]*len(args)+[BoolSort()]
                self.func_cache[key]=Function(name, *sorts)
            return self.func_cache[key](*args) if args else self.func_cache[key]()

print("FOL parser v11 ready (handles forall/exists/->/<->/not/and/or + arithmetic + string-literals)")


FOL parser v11 ready (handles forall/exists/->/<->/not/and/or + arithmetic + string-literals)


In [11]:
# ==================================================================
# STAGE 4 -- v12 Pipeline
# FOL-string Pass-2 + BoN Self-Consistency + Predicate Grounding
# + Confidence Gating (3-tier) + idx Filter + Unsat-Core Explanation
# ==================================================================
import re, time, json
from dataclasses import dataclass, field
from collections import Counter
from z3 import Solver, Not, Bool, sat, unsat
from vllm import SamplingParams

# v13.4: Qwen-CoT generation with LoRA v14 (separate from formalizer LoRA path)
def batch_generate_cot_bon(prompt_pairs, max_tokens, n=5, temperature=0.5, enable_thinking=False):
    """v13.6: Qwen-CoT Best-of-N with LoRA v14. Returns list[list[str]] (N cands/prompt)."""
    formatted = []
    for sys_msg, usr_msg in prompt_pairs:
        messages = [{"role": "system", "content": sys_msg},
                    {"role": "user", "content": usr_msg}]
        try:
            text = tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True,
                enable_thinking=enable_thinking)
        except TypeError:
            text = tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True)
        formatted.append(text)
    params = SamplingParams(temperature=temperature, top_p=0.95, max_tokens=max_tokens,
                            n=n, repetition_penalty=1.1)
    if COT_LORA_REQ is not None:
        outputs = llm.generate(formatted, params, lora_request=COT_LORA_REQ)
    else:
        outputs = llm.generate(formatted, params)
    outputs_sorted = sorted(outputs, key=lambda x: int(x.request_id))
    return [[o.text.strip() for o in out.outputs] for out in outputs_sorted]


def cited_premises(text):
    """Parse premise numbers Qwen cites ('premise 3', 'P3') -> 0-based index set."""
    nums = set()
    for m in re.finditer(r'[Pp]remise\s+(?:number\s+)?(\d+)', text or ''):
        nums.add(int(m.group(1)) - 1)
    for m in re.finditer(r'\bP(\d+)\b', text or ''):
        nums.add(int(m.group(1)) - 1)
    return nums


def compute_z3_core(g, sample, q_idx, rep_fol):
    """Unsat-core premise indices for a YN statement (for semantic-agreement feature)."""
    if not rep_fol:
        return set()
    try:
        stmt = FOLParser(g['fc'], g['cm'], g['ic']).parse(rep_fol)
    except Exception:
        return set()
    idx_list = sample.get('idx', [])
    if USE_IDX_FILTER and q_idx < len(idx_list) and isinstance(idx_list[q_idx], list) and idx_list[q_idx]:
        pwi = [(g['per_idx'][j-1], j-1) for j in idx_list[q_idx] if (j-1) in g['per_idx']]
        if not pwi:
            pwi = [(g['per_idx'][k], k) for k in g['per_idx']]
    else:
        pwi = [(g['per_idx'][k], k) for k in g['per_idx']]
    try:
        return set(unsat_core_premises(pwi, stmt))
    except Exception:
        return set()


def batch_generate_cot(prompt_pairs, max_tokens, enable_thinking=False):
    """Generate one Qwen-CoT response per prompt using v14 LoRA when available."""
    formatted = []
    for sys_msg, usr_msg in prompt_pairs:
        messages = [{"role": "system", "content": sys_msg},
                    {"role": "user", "content": usr_msg}]
        try:
            text = tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True,
                enable_thinking=enable_thinking)
        except TypeError:
            text = tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True)
        formatted.append(text)
    params = SamplingParams(temperature=0.1, top_p=0.95, max_tokens=max_tokens,
                            repetition_penalty=1.1)
    if COT_LORA_REQ is not None:
        outputs = llm.generate(formatted, params, lora_request=COT_LORA_REQ)
    else:
        outputs = llm.generate(formatted, params)
    outputs_sorted = sorted(outputs, key=lambda x: int(x.request_id))
    return [out.outputs[0].text.strip() for out in outputs_sorted]



@dataclass
class PipelineResult:
    sample_id: int
    status: str = "pending"
    z3_status: str = "pending"
    z3_compiled: int = 0
    z3_total: int = 0
    z3_attempts: int = 0
    z3_errors: list = field(default_factory=list)
    hallucination_warn: list = field(default_factory=list)
    local_ontology: list = field(default_factory=list)
    premises_ast: list = field(default_factory=list)
    question_fol: list = field(default_factory=list)
    predicted_answers: list = field(default_factory=list)
    ground_truth: list = field(default_factory=list)
    total_questions: int = 0
    correct_count: int = 0
    time_sec: float = 0.0
    error_log: list = field(default_factory=list)
    answer_source: list = field(default_factory=list)


# ---------------- Question type & embedded-FOL detection ----------------
def is_mc_question(q: str) -> bool:
    return bool(re.search(r'\n\s*[ABCD][\.\)]', q))


def extract_embedded_fol(q: str):
    """If a Yes/No question already contains a FOL Statement, return the FOL string."""
    if not re.search(r'[\u2200\u2203\u2192\u00ac\u2227\u2228\u2194]', q):
        return None
    if is_mc_question(q):
        return None
    parts = re.split(r'[Ss]tatement\s*:', q)
    if len(parts) > 1:
        cand = parts[-1].strip().strip("'\"").strip()
        # strip trailing punctuation like '?' or '.' if at end
        cand = re.sub(r'[\?\.]\s*$', '', cand).strip()
        return cand or None
    return None


# ---------------- Predicate grounding (exact, case-insensitive only) ----------------
def extract_predicates_used(fol_str: str):
    # Predicates are written with name DIRECTLY adjacent to '(' (no space).
    # Variables in '∀x (' have a space before '(' and won't match.
    return set(re.findall(r'([A-Za-z_]\w*)\(', fol_str))


def ground_predicates(fol_str: str, allowed: set):
    """Map predicate names to canonical names in `allowed` via exact case-insensitive match.
    Returns (grounded_str, ungrounded_set). Empty ungrounded_set => fully grounded."""
    used = extract_predicates_used(fol_str)
    ungrounded = set()
    out = fol_str
    allowed_lower = {p.lower(): p for p in allowed}
    for p in used:
        if p in allowed:
            continue
        canon = allowed_lower.get(p.lower())
        if canon and canon != p:
            out = re.sub(rf'\b{re.escape(p)}\b', canon, out)
        elif not canon:
            ungrounded.add(p)
    return out, ungrounded


# ---------------- Pass-2 prompts (FOL-string output) ----------------
PASS2_YN_SYSTEM = (
    # Declarative prompting inspired by SatLM (Ye et al., 2023): the LLM only
    # produces a formal target specification; Z3 is the reasoner.
    "You are a translator from natural language to First-Order Logic. "
    "Your ONLY job is to translate the question's statement into ONE FOL formula. "
    "STRICT RULES:\n"
    "- Do NOT solve the question. Do NOT reason. Do NOT output any answer (no Yes/No/Unknown).\n"
    "- Use ONLY the predicate names that appear in the premises shown below, verbatim.\n"
    "- Do NOT reverse implications: if a premise is A \u2192 B, do not write B \u2192 A.\n"
    "- Allowed symbols ONLY: \u2200 \u2203 \u2192 \u00ac \u2227 \u2228 \u2194.\n"
    "- Return the FOL formula on a single line. No quotes, no labels, no explanation."
)

PASS2_MC_SYSTEM = (
    # Declarative prompting inspired by SatLM (Ye et al., 2023).
    "You are a translator from natural language to First-Order Logic. "
    "Your ONLY job is to translate EACH of the 4 options into ONE FOL formula. "
    "STRICT RULES:\n"
    "- Do NOT solve. Do NOT reason. Do NOT pick or output an answer letter as the result.\n"
    "- Use ONLY the predicate names that appear in the premises shown below, verbatim.\n"
    "- Do NOT reverse implications: if a premise is A \u2192 B, do not write B \u2192 A.\n"
    "- Allowed symbols ONLY: \u2200 \u2203 \u2192 \u00ac \u2227 \u2228 \u2194.\n"
    "Output EXACTLY 4 lines (one formula per option):\n"
    "A: <formula>\nB: <formula>\nC: <formula>\nD: <formula>\n"
    "No explanation."
)


# ==================================================================
# Stage C.5 -- One-step solver-feedback REPAIR prompts (inspired by Logic-LM)
# Logic-LM (Pan et al., 2023): solver-error messages drive LLM self-refinement.
# Our adaptation: feedback is constrained to parser/Z3/grounding diagnostics
# only; NEVER includes the gold answer; NEVER says "your answer is wrong".
# ==================================================================
PASS2_YN_REPAIR_SYSTEM = (
    "You previously translated a Yes/No question into a FOL formula, but an "
    "automated FORMAL checker found an issue with the formula (not with any answer). "
    "Produce a corrected FOL formula that fixes ONLY the reported formal issue. "
    "STRICT RULES:\n"
    "- Do NOT solve or output any answer.\n"
    "- Use ONLY the predicate names listed; do not invent predicates.\n"
    "- Do NOT reverse implications.\n"
    "- Bind every variable with a quantifier (no free variables).\n"
    "- Allowed symbols ONLY: \u2200 \u2203 \u2192 \u00ac \u2227 \u2228 \u2194.\n"
    "- Return ONLY the corrected formula on a single line."
)
PASS2_MC_REPAIR_SYSTEM = (
    "You previously translated 4 options into FOL, but an automated FORMAL checker "
    "found an issue. Produce corrected FOL for the options, fixing ONLY the reported "
    "formal issue. Same strict rules as before; do not solve, do not output an answer letter. "
    "Output EXACTLY 4 lines: A: ...  B: ...  C: ...  D: ..."
)

def build_repair_feedback(diag_codes, allowed_preds, idx_preds=None):
    """Compose a feedback message from FORMAL diagnostics only (no gold answer)."""
    lines = ["The automated checker reported these FORMAL issues with your formula:"]
    code_msg = {
        "parse_fail":      "- The formula could not be parsed as valid FOL.",
        "ungrounded_pred": "- It uses predicate name(s) not present in the premises.",
        "free_var":        "- It contains a variable that is not bound by any quantifier.",
        "ignores_idx":     "- It does not use any of the premises relevant to this question.",
        "no_mc_entailed":  "- None of the 4 options is entailed by the premises.",
        "multi_mc_entailed":"- More than one option is entailed; the formalization may be too weak.",
        "z3_unknown_conf": "- The solver could not determine entailment from this formula.",
        "contradiction":   "- The formula contradicts the premises in an unexpected way.",
    }
    for code in diag_codes:
        if code in code_msg:
            lines.append(code_msg[code])
    lines.append("")
    lines.append("Allowed predicate names: " + ", ".join(sorted(allowed_preds))[:600])
    if idx_preds:
        lines.append("Premises most relevant to this question: " + ", ".join(str(j) for j in idx_preds))
    lines.append("")
    lines.append("Return a corrected FOL formula only.")
    return "\n".join(lines)



def _relevant_premise_fols(sample, q_idx):
    fols = sample.get('premises-FOL', [])
    idx_list = sample.get('idx', [])
    if USE_IDX_FILTER and q_idx < len(idx_list) and isinstance(idx_list[q_idx], list) and idx_list[q_idx]:
        rel = [fols[j-1] for j in idx_list[q_idx] if 0 <= j-1 < len(fols)]
        if rel:
            return rel
    return fols


def _ontology_str(preds: set, max_chars=600):
    return ", ".join(sorted(preds))[:max_chars]


def make_pass2_yn_user(sample, q, q_idx, preds):
    rel = _relevant_premise_fols(sample, q_idx)
    shots = "\n".join(f"- {f}" for f in rel)
    onto = _ontology_str(preds)
    return (f"Relevant premises (FOL syntax to mirror):\n{shots}\n\n"
            f"Available predicate names: {onto}\n\n"
            f"Question: {q}\n\nFormula:")


def make_pass2_mc_user(sample, q, q_idx, preds):
    rel = _relevant_premise_fols(sample, q_idx)
    shots = "\n".join(f"- {f}" for f in rel)
    onto = _ontology_str(preds)
    return (f"Relevant premises (FOL syntax to mirror):\n{shots}\n\n"
            f"Available predicate names: {onto}\n\n"
            f"Question (multiple choice):\n{q}\n\n"
            f"Write each option as a FOL formula:")


# ---------------- Parse Pass-2 outputs ----------------
def parse_yn_output(raw: str) -> str:
    """Extract a single FOL formula from a (possibly noisy) Yes/No Pass-2 output."""
    raw = (raw or "").strip()
    for line in reversed(raw.split('\n')):
        line = line.strip().strip('`').strip()
        line = re.sub(r'^(?:formula|fol|answer|statement)\s*[:=]\s*', '', line, flags=re.I)
        if re.search(r'[\u2200\u2203\u2192\u00ac\u2227\u2228\u2194]', line) or re.search(r'[A-Za-z_]\w*\s*\(', line):
            return line
    # last resort: last non-empty line
    lines = [l for l in raw.split('\n') if l.strip()]
    return lines[-1].strip() if lines else ""


def parse_mc_output(raw: str) -> dict:
    """Extract a dict {A,B,C,D} -> FOL string from a Pass-2 MC output."""
    out = {}
    for line in (raw or "").split('\n'):
        m = re.match(r'^\s*([ABCD])\s*[:\.\)]\s*(.+)$', line.strip())
        if m:
            f = m.group(2).strip().strip('`').strip()
            out[m.group(1)] = f
    return out


# ---------------- Premise FOL parsing ----------------
def parse_gold_premises(fol_strings):
    fc, cm, ic = {}, {}, {}
    exprs, per_idx = [], {}
    n_ok = 0
    for i, f in enumerate(fol_strings):
        try:
            e = FOLParser(fc, cm, ic).parse(f)
            exprs.append(e)
            per_idx[i] = e
            n_ok += 1
        except Exception:
            pass
    all_ok = (n_ok == len(fol_strings)) and len(fol_strings) > 0
    preds = set(k.rsplit('_', 1)[0] for k in fc.keys() if not k.startswith('__'))
    return dict(exprs=exprs, per_idx=per_idx, fc=fc, cm=cm, ic=ic,
                preds=preds, all_ok=all_ok, n_ok=n_ok, n_tot=len(fol_strings))


def _select_premises(g, sample, q_idx):
    if not USE_IDX_FILTER:
        return g['exprs']
    idx_list = sample.get('idx', [])
    if q_idx < len(idx_list) and isinstance(idx_list[q_idx], list) and idx_list[q_idx]:
        rel = [g['per_idx'][j-1] for j in idx_list[q_idx] if (j-1) in g['per_idx']]
        if rel:
            return rel
    return g['exprs']


# ---------------- Entailment ----------------
def _solver_with(exprs):
    s = Solver(); s.set('timeout', Z3_SOLVER_TIMEOUT)
    for e in exprs:
        s.add(e)
    return s


def entail_yn(premise_exprs, stmt):
    s = _solver_with(premise_exprs)
    s.push(); s.add(Not(stmt))
    try:
        r1 = s.check()
    except Exception:
        r1 = None
    s.pop()
    if r1 == unsat:
        return 'Yes'
    s.push(); s.add(stmt)
    try:
        r2 = s.check()
    except Exception:
        r2 = None
    s.pop()
    if r2 == unsat:
        return 'No'
    return 'Unknown'


def entail_mc(premise_exprs, options_exprs):
    if not options_exprs:
        return 'Unknown'
    entailed, consistent, contradicted = [], [], []
    for lab, oe in options_exprs.items():
        s = _solver_with(premise_exprs)
        s.push(); s.add(Not(oe))
        try: r1 = s.check()
        except Exception: r1 = None
        s.pop()
        if r1 == unsat:
            entailed.append(lab)
        s.push(); s.add(oe)
        try: r2 = s.check()
        except Exception: r2 = None
        s.pop()
        if r2 == unsat:
            contradicted.append(lab)
        elif r2 == sat:
            consistent.append(lab)
    if len(entailed) == 1:
        return entailed[0]
    if len(entailed) > 1:
        return entailed[0]
    nonc = [l for l in options_exprs if l in consistent and l not in contradicted]
    if len(nonc) == 1:
        return nonc[0]
    if len(consistent) == 1:
        return consistent[0]
    return 'Unknown'


# ---------------- Unsat-core explanation (XAI deliverable) ----------------
def unsat_core_premises(premise_exprs_with_idx, stmt):
    """premise_exprs_with_idx: list of (premise_z3_expr, original_index).
    Returns list of original indices whose premises were used in the proof, or []."""
    s = Solver(); s.set('timeout', Z3_SOLVER_TIMEOUT)
    bools = []
    for k, (e, orig_i) in enumerate(premise_exprs_with_idx):
        b = Bool(f'__core_p{k}')
        s.assert_and_track(e, b)
        bools.append((b, orig_i))
    s.add(Not(stmt))
    try:
        if s.check() == unsat:
            core_strs = set(str(c) for c in s.unsat_core())
            return [orig_i for (b, orig_i) in bools if str(b) in core_strs]
    except Exception:
        pass
    return []


# ---------------- Self-consistency vote ----------------
def sc_vote_yn(premise_exprs, candidates, preds, fc, cm, ic):
    """Vote over Yes/No/Unknown from BoN FOL-string candidates."""
    votes = Counter()
    n_parsed = 0
    grounded_fols = []
    for cand in candidates:
        if not cand:
            votes['_empty'] += 1
            continue
        grounded, ung = ground_predicates(cand, preds)
        if ung:
            votes['_ungrounded'] += 1
            continue
        try:
            stmt = FOLParser(fc, cm, ic).parse(grounded)
        except Exception:
            votes['_parse_err'] += 1
            continue
        n_parsed += 1
        grounded_fols.append((grounded, stmt))
        votes[entail_yn(premise_exprs, stmt)] += 1
    return votes, n_parsed, grounded_fols


def sc_vote_mc(premise_exprs, candidate_sets, preds, fc, cm, ic):
    """Vote over A/B/C/D/Unknown from BoN candidate sets."""
    votes = Counter()
    n_parsed = 0
    for cand in candidate_sets:
        if not isinstance(cand, dict) or len(cand) < 2:
            votes['_empty'] += 1
            continue
        opt_exprs = {}
        ok = True
        for lab in 'ABCD':
            if lab not in cand:
                continue
            g, ung = ground_predicates(cand[lab], preds)
            if ung:
                ok = False; break
            try:
                opt_exprs[lab] = FOLParser(fc, cm, ic).parse(g)
            except Exception:
                ok = False; break
        if not ok or len(opt_exprs) < 2:
            votes['_failed'] += 1
            continue
        n_parsed += 1
        votes[entail_mc(premise_exprs, opt_exprs)] += 1
    return votes, n_parsed


def gate_decision(votes: Counter, total_candidates: int):
    """3-tier gate: returns (best_answer, confidence, tier).
    tier in {'z3_trust', 'z3_arbiter', 'fallback'}."""
    real = {k: v for k, v in votes.items() if not k.startswith('_')}
    if not real:
        return None, 0.0, 'fallback'
    total = sum(real.values())
    ans, c = max(real.items(), key=lambda x: x[1])
    conf = c / total
    grounded_frac = total / max(total_candidates, 1)
    definite = ans not in (None, 'Unknown')
    if definite and conf >= SC_CONFIDENCE_TAU and grounded_frac >= GROUNDED_FRAC_TAU:
        return ans, conf, 'z3_trust'
    if definite and conf >= 0.5:
        return ans, conf, 'z3_arbiter'
    return ans, conf, 'fallback'


# ---------------- BoN generation helper ----------------
def batch_generate_n(prompt_pairs, max_tokens, n=5, temperature=0.7, enable_thinking=False):
    """Generate N samples per prompt using vLLM n parameter. Returns list[list[str]]."""
    formatted = []
    for sys_msg, usr_msg in prompt_pairs:
        messages = [{"role": "system", "content": sys_msg},
                    {"role": "user", "content": usr_msg}]
        try:
            text = tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True,
                enable_thinking=enable_thinking)
        except TypeError:
            text = tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True)
        formatted.append(text)
    params = SamplingParams(
        temperature=temperature, top_p=0.95, max_tokens=max_tokens, n=n,
        repetition_penalty=1.1,
    )
    outputs = llm.generate(formatted, params)
    outputs_sorted = sorted(outputs, key=lambda x: int(x.request_id))
    return [[o.text.strip() for o in out.outputs] for out in outputs_sorted]


# ---------------- Answer-fallback prompt (reuse from utils) ----------------
def _make_answer_user(sample, question):
    lines = ["### Premises"]
    for i, p in enumerate(sample.get('premises-NL', [])):
        lines.append(f"P{i+1}. {p}")
    return "\n".join(lines) + f"\n\n### Question\n{question}"


def _make_informed_answer_user(sample, question, z3_hint, z3_conf, votes_summary, has_z3):
    """v12.1 -- include premise FOL and Z3 evidence so Qwen can reason informedly."""
    nls = sample.get('premises-NL', [])
    fols = sample.get('premises-FOL', [])
    lines = ["### Premises (NL + FOL)"]
    for i, p in enumerate(nls):
        lines.append(f"P{i+1}. {p}")
        if i < len(fols) and fols[i]:
            lines.append(f"      FOL: {fols[i]}")
    out = "\n".join(lines) + f"\n\n### Question\n{question}"
    if has_z3 and z3_hint:
        out += "\n\n### Symbolic evidence (Z3 evaluated 5 formalizations)"
        out += f"\n- Vote distribution: {votes_summary or '(empty)'}"
        out += f"\n- Z3's tentative answer: {z3_hint} (confidence {z3_conf:.2f})"
        out += ("\n- Note: Z3 may over-claim entailment on statements that are "
                "converses/inverses of premises or have unscoped free variables. "
                "Verify carefully by reasoning step-by-step.")
    return out




# ============================== STRUCTURAL FEATURES ==============================
def _split_top_implication(fol_str):
    """Split a FOL string on its top-level '->' (outside any parens). Returns (ante, cons) or None."""
    depth = 0
    for i, ch in enumerate(fol_str):
        if ch == '(':
            depth += 1
        elif ch == ')':
            depth -= 1
        elif ch == '\u2192' and depth == 0:
            return fol_str[:i], fol_str[i+1:]
    # strip one outer quantifier+paren layer and retry once
    m = re.match(r'^\s*[\u2200\u2203]\w+\s*\((.*)\)\s*$', fol_str)
    if m:
        return _split_top_implication(m.group(1))
    return None


def is_converse_or_inverse(stmt_fol, premise_fols):
    """Heuristic: stmt is A->B; some premise is B->A (converse) or shares
    swapped antecedent/consequent predicate sets (inverse via negation)."""
    s = _split_top_implication(stmt_fol)
    if not s:
        return False
    sa = extract_predicates_used(s[0])
    sb = extract_predicates_used(s[1])
    if not sa or not sb:
        return False
    for pf in premise_fols:
        p = _split_top_implication(pf)
        if not p:
            continue
        pa = extract_predicates_used(p[0])
        pb = extract_predicates_used(p[1])
        if sa == pb and sb == pa and sa != sb:
            return True
    return False


def _has_free_var(fol_str):
    """Variable appears in an atom but no matching quantifier binds it (rough)."""
    quantified = set(re.findall(r'[\u2200\u2203](\w+)', fol_str))
    # vars used as args: lowercase tokens inside parens
    used_vars = set(re.findall(r'[\(,]\s*([a-z]\w*)\s*[\),]', fol_str))
    return bool(used_vars - quantified)


def structural_features(stmt_fol, premise_fols):
    if not stmt_fol:
        return dict(has_free_var=0, is_converse=0, n_quantifiers=0, stmt_len=0)
    return dict(
        has_free_var=int(_has_free_var(stmt_fol)),
        is_converse=int(is_converse_or_inverse(stmt_fol, premise_fols)),
        n_quantifiers=stmt_fol.count('\u2200') + stmt_fol.count('\u2203'),
        stmt_len=len(stmt_fol),
    )


def pick_representative_fol(cands, preds, fc, cm, ic, prem):
    """Return one grounded, parseable FOL string from BoN candidates (for features/explanation)."""
    for c in cands:
        if not c:
            continue
        g, ung = ground_predicates(c, preds)
        if ung:
            continue
        try:
            FOLParser(fc, cm, ic).parse(g)
            return g
        except Exception:
            continue
    return ""


# ============================== FEATURE-EXTRACTION PIPELINE ==============================
def build_features(samples, dataset_name="Dataset"):
    """Run Z3 BoN + entailment AND full Qwen-CoT on EVERY question.
    Returns (records, results_skeleton)."""
    t0 = time.time()
    N = len(samples)
    results = [PipelineResult(sample_id=i,
                              ground_truth=s.get('answers', []),
                              total_questions=len(s.get('questions', [])))
               for i, s in enumerate(samples)]

    # ---- Stage A: premises ----
    print("\n" + "=" * 65 + "\n  STAGE A -- Premise FOL parse\n" + "=" * 65)
    gold = {}
    for i, s in enumerate(samples):
        g = parse_gold_premises(s.get('premises-FOL', []))
        gold[i] = g
        results[i].z3_compiled = g['n_ok']; results[i].z3_total = g['n_tot']
        results[i].z3_status = 'sat' if (g['exprs'] and (g['all_ok'] or g['n_ok'] >= max(1, g['n_tot']-1))) else 'no_ast'
    print("  Premise status:", dict(Counter(r.z3_status for r in results)))

    # ---- Stage B: dispatch ----
    embedded, yn_tasks, mc_tasks = [], [], []
    all_questions = []   # (i, q_idx, q, q_type)
    for i, s in enumerate(samples):
        for q_idx, q in enumerate(s.get('questions', [])):
            qt = 'mc' if is_mc_question(q) else 'yes_no'
            all_questions.append((i, q_idx, q, qt))
            if results[i].z3_status != 'sat':
                continue
            emb = extract_embedded_fol(q)
            if emb:
                embedded.append((i, q_idx, emb))
            elif qt == 'mc':
                mc_tasks.append((i, q_idx, q))
            else:
                yn_tasks.append((i, q_idx, q))
    print(f"  embedded:{len(embedded)} yn:{len(yn_tasks)} mc:{len(mc_tasks)} total_q:{len(all_questions)}")

    # ---- Stage C: Z3 Pass-2 BoN ----
    print("\n" + "=" * 65 + f"\n  STAGE C -- Z3 Pass-2 BoN (N={BON_N_QFORMALIZE})\n" + "=" * 65)
    yn_cands, mc_cands = {}, {}
    if yn_tasks:
        prompts = [(PASS2_YN_SYSTEM, make_pass2_yn_user(samples[i], q, q_idx, gold[i]['preds']))
                   for (i, q_idx, q) in yn_tasks]
        gen = batch_generate_n(prompts, MAX_PASS2_TOKENS_YN, n=BON_N_QFORMALIZE, temperature=BON_TEMP, enable_thinking=False)
        for k, c in enumerate(gen):
            i, q_idx, _ = yn_tasks[k]; yn_cands[(i, q_idx)] = [parse_yn_output(x) for x in c]
    if mc_tasks:
        prompts = [(PASS2_MC_SYSTEM, make_pass2_mc_user(samples[i], q, q_idx, gold[i]['preds']))
                   for (i, q_idx, q) in mc_tasks]
        gen = batch_generate_n(prompts, MAX_PASS2_TOKENS_MC, n=BON_N_QFORMALIZE, temperature=BON_TEMP, enable_thinking=False)
        for k, c in enumerate(gen):
            i, q_idx, _ = mc_tasks[k]; mc_cands[(i, q_idx)] = [parse_mc_output(x) for x in c]
    print(f"  Z3 BoN done. yn={len(yn_cands)} mc={len(mc_cands)}")

    # ---- Stage C2: compute Z3 signals per question ----
    z3_sig = {}   # (i,q_idx) -> dict
    for (i, q_idx, fol) in embedded:
        g = gold[i]; prem = _select_premises(g, samples[i], q_idx)
        gg, ung = ground_predicates(fol, g['preds'])
        if ung:
            z3_sig[(i, q_idx)] = dict(answer='Unknown', conf=0.0, grounded=0.0, definite=False,
                                      spread=0, rep_fol='', source='embed_ungrounded'); continue
        try:
            stmt = FOLParser(g['fc'], g['cm'], g['ic']).parse(gg)
            ans = entail_yn(prem, stmt)
            z3_sig[(i, q_idx)] = dict(answer=ans, conf=1.0, grounded=1.0, definite=(ans != 'Unknown'),
                                      spread=1, rep_fol=gg, source='embed')
        except Exception:
            z3_sig[(i, q_idx)] = dict(answer='Unknown', conf=0.0, grounded=0.0, definite=False,
                                      spread=0, rep_fol='', source='embed_err')
    for (i, q_idx, q) in yn_tasks:
        g = gold[i]; prem = _select_premises(g, samples[i], q_idx)
        cands = yn_cands.get((i, q_idx), [])
        votes, n_parsed, gfols = sc_vote_yn(prem, cands, g['preds'], g['fc'], g['cm'], g['ic'])
        ans, conf, tier = gate_decision(votes, len(cands))
        real = {k: v for k, v in votes.items() if not k.startswith('_')}
        z3_sig[(i, q_idx)] = dict(answer=ans or 'Unknown', conf=conf,
                                  grounded=(n_parsed/max(len(cands), 1)),
                                  definite=(ans not in (None, 'Unknown')),
                                  spread=len(real),
                                  rep_fol=(gfols[0][0] if gfols else pick_representative_fol(cands, g['preds'], g['fc'], g['cm'], g['ic'], prem)),
                                  source='yn')
    for (i, q_idx, q) in mc_tasks:
        g = gold[i]; prem = _select_premises(g, samples[i], q_idx)
        cands = mc_cands.get((i, q_idx), [])
        votes, n_parsed = sc_vote_mc(prem, cands, g['preds'], g['fc'], g['cm'], g['ic'])
        ans, conf, tier = gate_decision(votes, len(cands))
        real = {k: v for k, v in votes.items() if not k.startswith('_')}
        z3_sig[(i, q_idx)] = dict(answer=ans or 'Unknown', conf=conf,
                                  grounded=(n_parsed/max(len(cands), 1)),
                                  definite=(ans not in (None, 'Unknown')),
                                  spread=len(real), rep_fol='', source='mc')

    # ---- Stage D: Qwen-CoT BoN + Self-Consistency (v13.6 Option B) ----
    print("\n" + "=" * 65 + f"\n  STAGE D -- Qwen-CoT BoN (N={QWEN_BON_N}, {len(all_questions)} questions)\n" + "=" * 65)
    cot_prompts = [(V14_COT_SYSTEM, _make_answer_user(samples[i], q)) for (i, q_idx, q, qt) in all_questions]
    cot_gen = batch_generate_cot_bon(cot_prompts, COT_MAX_TOKENS, n=QWEN_BON_N,
                                     temperature=QWEN_BON_TEMP, enable_thinking=False)
    qwen_map = {}   # (i,q_idx) -> dict(answer, conf, spread, text)
    def _norm(a):
        a = str(a).strip()
        return a.upper() if a.lower() in ('yes', 'no', 'unknown') else a.upper()
    for k, cands in enumerate(cot_gen):
        i, q_idx, q, qt = all_questions[k]
        ans_list, text_by_ans = [], {}
        for c in cands:
            a = extract_final_answer(c)
            if a == 'UNPARSEABLE':
                a = safe_json(c).get('answer', 'Unknown')
            a = _norm(a)
            ans_list.append(a)
            text_by_ans.setdefault(a, c)
        vote = Counter(ans_list)
        if vote:
            top_ans, top_n = vote.most_common(1)[0]
            conf = top_n / max(len(ans_list), 1)
            spread = len(vote)
            text = text_by_ans.get(top_ans, cands[0] if cands else "")
        else:
            top_ans, conf, spread, text = 'Unknown', 0.0, 0, ""
        qwen_map[(i, q_idx)] = dict(answer=top_ans, conf=conf, spread=spread, text=text)
    # backward-compat alias
    qwen_ans_map = {k: v['answer'] for k, v in qwen_map.items()}

    # ---- Stage E: assemble feature records ----
    records = []
    for (i, q_idx, q, qt) in all_questions:
        gt = samples[i].get('answers', [])
        gold_ans = gt[q_idx] if q_idx < len(gt) else '?'
        z = z3_sig.get((i, q_idx), dict(answer='Unknown', conf=0.0, grounded=0.0,
                                        definite=False, spread=0, rep_fol='', source='no_premise'))
        qm = qwen_map.get((i, q_idx), dict(answer='Unknown', conf=0.0, spread=0, text=''))
        qa = qm['answer']
        idx_list = samples[i].get('idx', [])
        n_idx = len(idx_list[q_idx]) if (q_idx < len(idx_list) and isinstance(idx_list[q_idx], list)) else 0
        sf = structural_features(z['rep_fol'], samples[i].get('premises-FOL', []))
        agree = int(str(z['answer']).strip().upper() == str(qa).strip().upper())
        # v13.6 Option C: semantic agreement between Z3 unsat-core and Qwen citations
        qwen_cited = cited_premises(qm['text'])
        z3_core = compute_z3_core(gold[i], samples[i], q_idx, z['rep_fol'])
        if z3_core or qwen_cited:
            inter = len(z3_core & qwen_cited); uni = len(z3_core | qwen_cited)
            jacc = inter / uni if uni else 0.0
        else:
            jacc = 0.0
        records.append(dict(
            sample_id=i, q_idx=q_idx, q_type=qt, gold=str(gold_ans),
            z3_answer=str(z['answer']), z3_conf=round(float(z['conf']), 4),
            z3_grounded=round(float(z['grounded']), 4), z3_definite=int(z['definite']),
            z3_spread=int(z['spread']), z3_source=z['source'],
            qwen_answer=str(qa), z3_qwen_agree=agree,
            qwen_sc_conf=round(float(qm['conf']), 4), qwen_vote_spread=int(qm['spread']),
            qwen_n_cited=len(qwen_cited), z3_qwen_jaccard=round(float(jacc), 4),
            q_type_mc=int(qt == 'mc'), n_idx_premises=n_idx,
            **sf,
            z3_correct=int(str(z['answer']).strip().upper() == str(gold_ans).strip().upper()),
            qwen_correct=int(str(qa).strip().upper() == str(gold_ans).strip().upper()),
        ))

    print(f"\n  Built {len(records)} feature records in {time.time()-t0:.1f}s")
    return records, results, gold


print("v13.6 feature-extraction pipeline loaded (Qwen BoN-SC + premise-overlap features)")


v13.6 feature-extraction pipeline loaded (Qwen BoN-SC + premise-overlap features)


In [12]:
import json, time, random, os
import numpy as np
from collections import defaultdict, Counter

# Load cache or rebuild (same as before)
records = None
if LOAD_FEATURES_CACHE and os.path.exists(FEATURES_CACHE_PATH) and not FORCE_REBUILD:
    try:
        records = json.load(open(FEATURES_CACHE_PATH, encoding="utf-8"))
        print(f"Loaded {len(records)} cached feature records from {FEATURES_CACHE_PATH}")
    except Exception as e:
        print(f"Cache load failed ({e}); will rebuild"); records = None
if records is None:
    logic_samples = load_dataset(DATASET_PATH, is_physics=False)
    print(f"Building features from scratch on {len(logic_samples)} samples...")
    if len(logic_samples) == 0:
        raise RuntimeError(f"Dataset returned 0 samples from DATASET_PATH={DATASET_PATH}")
    records, _, _gold_parse = build_features(logic_samples, "Logic")
    globals()["_GOLD_PARSE"] = _gold_parse
    globals()["_LOGIC_SAMPLES"] = logic_samples
    json.dump(records, open(FEATURES_CACHE_PATH, "w", encoding="utf-8"), ensure_ascii=False, indent=1)
    print(f"Saved cache: {FEATURES_CACHE_PATH}")

n_samples = max(r["sample_id"] for r in records) + 1
print(f"Feature records: {len(records)}  |  samples: {n_samples}")

# Deterministic train/val split BY SAMPLE
rng = random.Random(CAL_SEED)
ids = list(range(n_samples)); rng.shuffle(ids)
n_tr = int(n_samples * CAL_TRAIN_RATIO)
train_ids = set(ids[:n_tr]); val_ids = set(ids[n_tr:])
train_recs = [r for r in records if r["sample_id"] in train_ids]
val_recs   = [r for r in records if r["sample_id"] in val_ids]
print(f"Split: train={len(train_recs)} q ({len(train_ids)} samples) | val={len(val_recs)} q ({len(val_ids)} samples)")

print("\nBuild + split done. Run Stage C.5 (next cell) BEFORE the comparison cell.")


Building features from scratch on 318 samples...

  STAGE A -- Premise FOL parse
  Premise status: {'sat': 316, 'no_ast': 2}
  embedded:44 yn:296 mc:286 total_q:630

  STAGE C -- Z3 Pass-2 BoN (N=5)


Rendering prompts:   0%|          | 0/296 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1480 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

(Worker_TP0 pid=265) WARNING 06-06 01:08:25 [jit_monitor.py:103] Triton kernel JIT compilation during inference: kernel_unified_attention. This causes a latency spike; consider extending warmup to cover this shape/config.
(Worker_TP0 pid=265) WARNING 06-06 01:08:27 [jit_monitor.py:103] Triton kernel JIT compilation during inference: reduce_segments. This causes a latency spike; consider extending warmup to cover this shape/config.
(Worker_TP0 pid=265) WARNING 06-06 01:08:32 [jit_monitor.py:103] Triton kernel JIT compilation during inference: _topk_topp_kernel. This causes a latency spike; consider extending warmup to cover this shape/config.


Processed prompts: 100%|██████████| 1480/1480 [01:16<00:00, 19.30it/s, est. speed input: 4830.20 toks/s, output: 450.16 toks/s]


...

WARNING 06-06 01:13:51 [input_processor.py:149] vLLM has deprecated support for supporting different tokenizers for different LoRAs. By default, vLLM uses base model's tokenizer. If you are using a LoRA with its own tokenizer, consider specifying `--tokenizer [lora_path]` to use the LoRA tokenizer.


Processed prompts:   0%|          | 0/3150 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

(Worker_TP0 pid=265) WARNING 06-06 01:13:54 [jit_monitor.py:103] Triton kernel JIT compilation during inference: _lora_expand_kernel. This causes a latency spike; consider extending warmup to cover this shape/config.
(Worker_TP0 pid=265) WARNING 06-06 01:13:55 [jit_monitor.py:103] Triton kernel JIT compilation during inference: _lora_shrink_kernel. This causes a latency spike; consider extending warmup to cover this shape/config.


Processed prompts: 100%|██████████| 3150/3150 [09:57<00:00,  5.27it/s, est. speed input: 1826.91 toks/s, output: 316.31 toks/s]



  Built 630 feature records in 928.7s
Saved cache: /kaggle/working/pipeline_features_16_1.json
Feature records: 630  |  samples: 318
Split: train=504 q (254 samples) | val=126 q (64 samples)

Build + split done. Run Stage C.5 (next cell) BEFORE the comparison cell.


In [14]:
# ==================================================================
# UTILS -- answer normalization helper
# ==================================================================
def _U(x):
    """Normalize answer string for comparison."""
    if x is None:
        return ""
    s = str(x).strip()
    up = s.upper()

    if up in ("YES", "Y", "TRUE"):
        return "YES"
    if up in ("NO", "N", "FALSE"):
        return "NO"
    if up in ("UNKNOWN", "UNK", "NOT ENOUGH INFORMATION", "CANNOT BE DETERMINED"):
        return "UNKNOWN"
    if up in ("A", "B", "C", "D"):
        return up
    return up

In [15]:
# ==================================================================
# STAGE C.5 -- One-step solver-feedback REPAIR (inspired by Logic-LM)
# Runs AFTER build_features. Re-formalizes ONLY weak-signal YN questions,
# re-checks Z3, and updates the record in place if the repair is grounded
# AND yields a definite verdict. MAX_REPAIR_ROUNDS=1 for Kaggle runtime.
#
# Attribution: solver-error self-refinement is from Logic-LM (Pan et al. 2023).
# Our adaptation: feedback restricted to parser/Z3/grounding diagnostics under
# gold premises-FOL + exact grounding + idx_premises; no gold answer leaked.
# ==================================================================
ENABLE_REPAIR    = True       # set False to skip Stage C.5
MAX_REPAIR_ROUNDS = 1

def _weak_signal(r):
    """Trigger repair when the symbolic signal is formally weak."""
    if r["q_type"] == "mc":
        return False  # keep MC repair as future work (riskier); focus YN
    # Z3 could not decide, but Qwen is confidently Yes/No -> possible bad formalization
    if not r["z3_definite"] and _U(r["qwen_answer"]) in ("YES", "NO"):
        if r.get("qwen_sc_conf", 0) >= 0.8:
            return True
    # grounded fraction very low -> formula likely used wrong predicates
    if r["z3_grounded"] < 0.4:
        return True
    return False

if ENABLE_REPAIR and "_GOLD_PARSE" in globals() and "_LOGIC_SAMPLES" in globals():
    gold_parse = _GOLD_PARSE
    samples    = _LOGIC_SAMPLES
    rec_by_key = {(r["sample_id"], r["q_idx"]): r for r in records}

    # Collect weak YN questions
    to_repair = []
    for r in records:
        if _weak_signal(r):
            to_repair.append(r)
    print(f"Stage C.5: {len(to_repair)} weak-signal YN questions flagged for 1-step repair")

    if to_repair:
        # Build repair prompts with FORMAL diagnostics only
        repair_prompts = []
        meta = []
        for r in to_repair:
            i, q_idx = r["sample_id"], r["q_idx"]
            g = gold_parse.get(i, {})
            preds = g.get("preds", set())
            q_text = samples[i]["questions"][q_idx]
            # Determine diagnostic codes
            diag = []
            if r["z3_grounded"] < 0.4:        diag.append("ungrounded_pred")
            if not r["z3_definite"]:           diag.append("z3_unknown_conf")
            if r.get("has_free_var", 0):       diag.append("free_var")
            idx_list = samples[i].get("idx", [])
            idx_pr = idx_list[q_idx] if (q_idx < len(idx_list) and isinstance(idx_list[q_idx], list)) else None
            fb = build_repair_feedback(diag or ["z3_unknown_conf"], preds, idx_pr)
            rel = _relevant_premise_fols(samples[i], q_idx)
            shots = "\n".join(f"- {f}" for f in rel)
            user = (f"Relevant premises (FOL):\n{shots}\n\n"
                    f"Question: {q_text}\n\n"
                    f"Your previous formula had issues:\n{fb}\n\nCorrected formula:")
            repair_prompts.append((PASS2_YN_REPAIR_SYSTEM, user))
            meta.append((r, g))

        # Single batched generation (1 round)
        reps = batch_generate_n(repair_prompts, MAX_PASS2_TOKENS_YN, n=1,
                                temperature=0.3, enable_thinking=False)
        n_fixed = 0
        for (r, g), cand_list in zip(meta, reps):
            cand = parse_yn_output(cand_list[0]) if cand_list else ""
            if not cand:
                continue
            grounded, ung = ground_predicates(cand, g.get("preds", set()))
            if ung:
                continue  # still ungrounded -> reject repair
            try:
                stmt = FOLParser(g["fc"], g["cm"], g["ic"]).parse(grounded)
            except Exception:
                continue
            i, q_idx = r["sample_id"], r["q_idx"]
            prem = _select_premises(g, samples[i], q_idx)
            new_ans = entail_yn(prem, stmt)
            # Accept repair ONLY if it now yields a definite verdict
            if new_ans in ("Yes", "No"):
                r["z3_answer"]   = new_ans
                r["z3_definite"] = 1
                r["z3_grounded"] = 1.0
                r["z3_conf"]     = max(r["z3_conf"], 0.8)
                r["z3_source"]   = "repair"
                r["z3_correct"]  = int(_U(new_ans) == _U(r["gold"]))
                r["z3_qwen_agree"] = int(_U(new_ans) == _U(r["qwen_answer"]))
                n_fixed += 1
        print(f"Stage C.5: {n_fixed}/{len(to_repair)} questions repaired to a definite Z3 verdict")
        # Refresh train/val record views (they reference the same dicts, so already updated)
    else:
        print("Stage C.5: nothing to repair")
else:
    print("Stage C.5 skipped (ENABLE_REPAIR=False or features loaded from cache without gold parse)")


Stage C.5: 98 weak-signal YN questions flagged for 1-step repair


Rendering prompts:   0%|          | 0/98 [00:00<?, ?it/s]

Processed prompts: 100%|██████████| 98/98 [00:36<00:00,  2.65it/s, est. speed input: 771.29 toks/s, output: 88.23 toks/s]  


Stage C.5: 19/98 questions repaired to a definite Z3 verdict


In [16]:
# ==================================================================
# DECISION + METRICS + COMPARISON  (run AFTER Stage C.5 repair)
# ==================================================================
import numpy as np
from collections import defaultdict, Counter



# ==================================================================
# METRICS -- accuracy + macro-F1 + per-class F1 (No-F1, Unknown-F1)
# ==================================================================
def _U(x): return str(x).strip().upper()

def compute_metrics(recs, decider, **kw):
    preds = [_U(decider(r, **kw)) for r in recs]
    golds = [_U(r["gold"]) for r in recs]
    labels = sorted(set(golds))
    per = {}
    for lab in labels:
        tp = sum(1 for p,g in zip(preds,golds) if p==lab and g==lab)
        fp = sum(1 for p,g in zip(preds,golds) if p==lab and g!=lab)
        fn = sum(1 for p,g in zip(preds,golds) if p!=lab and g==lab)
        prec = tp/(tp+fp) if (tp+fp) else 0.0
        rec  = tp/(tp+fn) if (tp+fn) else 0.0
        f1   = 2*prec*rec/(prec+rec) if (prec+rec) else 0.0
        per[lab] = dict(f1=f1, prec=prec, rec=rec, n=sum(1 for g in golds if g==lab))
    acc = sum(1 for p,g in zip(preds,golds) if p==g)/len(recs) if recs else 0.0
    macro_f1 = np.mean([per[l]["f1"] for l in labels]) if labels else 0.0
    n_tot = len(golds)
    weighted_f1 = sum(per[l]["f1"]*per[l]["n"] for l in labels)/n_tot if n_tot else 0.0
    return dict(acc=acc, macro_f1=macro_f1, weighted_f1=weighted_f1, per=per, preds=preds, golds=golds)


# ==================================================================
# DECIDERS -- rule-based only (NO LGBM in main path)
# ==================================================================
def dec_pure_qwen(r): return r["qwen_answer"]
def dec_pure_z3(r):   return r["z3_answer"] if r["z3_definite"] else r["qwen_answer"]
def dec_z3_first(r):
    if r["z3_definite"] and r["z3_conf"]>=SC_CONFIDENCE_TAU and r["z3_grounded"]>=GROUNDED_FRAC_TAU:
        return r["z3_answer"]
    return r["qwen_answer"]
def dec_parallel_hybrid(r):
    z,q = r["z3_answer"], r["qwen_answer"]
    if not r["z3_definite"]: return q
    if _U(z)==_U(q): return z
    if r["is_converse"] or r["has_free_var"]: return q
    if r["z3_conf"]>=PH_HIGH_CONF: return z
    return q
def dec_conf_hybrid(r):
    if not r["z3_definite"]: return r["qwen_answer"]
    if _U(r["z3_answer"])==_U(r["qwen_answer"]): return r["z3_answer"]
    if r.get("qwen_sc_conf",0)>=QWEN_CONF_TRUST and r.get("qwen_sc_conf",0)>r["z3_conf"]:
        return r["qwen_answer"]
    return r["z3_answer"]
def dec_oracle(r): return r["z3_answer"] if r["z3_correct"] else r["qwen_answer"]

# --- v16 ANTI-OVERCLAIM deciders (chống Yes-bias / underfit No+Unknown) ---
# Y tuong: khi Z3 KHONG entail duoc (def=0) NHUNG formalize tot (grounded cao),
# kha nang cao la cau that su Unknown -> tin Z3's Unknown thay vi Qwen overclaim.
# GROUND_TAU duoc tune tren TRAIN, validate tren VAL + 5-fold OOF.
def make_anti_overclaim(ground_tau, only_counter_yes=True):
    """Class-aware anti-overclaim (reviewer note #4). Rules:
    - Z3 definite: trust Z3.
    - MC: only override with Z3 when exactly one option entailed AND grounding high
      (z3_definite already encodes 'exactly one entailed' from entail_mc).
    - If Qwen predicts Unknown with high self-consistency, do NOT override (it's likely right).
    - If Qwen predicts Yes with weak grounding-backed Z3-Unknown, treat as overclaim -> Unknown.
    """
    def _dec(r):
        if r["z3_definite"]:
            return r["z3_answer"]
        # Qwen already says Unknown with high SC -> respect it (don't fight)
        if _U(r["qwen_answer"]) == "UNKNOWN" and r.get("qwen_sc_conf",0) >= QWEN_CONF_TRUST:
            return r["qwen_answer"]
        # Anti-overclaim: Z3 well-grounded but non-entailing (Unknown), Qwen overclaiming
        if r["z3_grounded"] >= ground_tau and _U(r["z3_answer"]) == "UNKNOWN":
            if only_counter_yes:
                if _U(r["qwen_answer"]) == "YES":     # counter the dominant Yes-bias
                    return "Unknown"
                return r["qwen_answer"]
            return "Unknown"
        return r["qwen_answer"]
    return _dec


# ==================================================================
# TUNE anti-overclaim threshold on TRAIN ONLY (defensible)
# ==================================================================
print("\n" + "="*70)
print("  Tuning anti-overclaim GROUND_TAU on TRAIN (counter Yes-bias)")
print("="*70)
best_tau, best_train_acc = 0.8, 0.0
for tau in [x/100 for x in range(50, 101, 5)]:
    m = compute_metrics(train_recs, make_anti_overclaim(tau, only_counter_yes=True))
    # uu tien macro-F1 de can bang class (khong chi accuracy)
    score = 0.5*m["acc"] + 0.5*m["macro_f1"]
    if score > best_train_acc:
        best_train_acc, best_tau = score, tau
print(f"  Best GROUND_TAU = {best_tau:.2f} (train blended score {best_train_acc:.3f})")


# ==================================================================
# 5-FOLD OOF for anti-overclaim (report mean +/- std -- defensible)
# ==================================================================
print("\n" + "="*70)
print("  5-fold OOF: anti-overclaim (sample-level folds)")
print("="*70)
_ids = sorted(set(r["sample_id"] for r in records))
random.Random(CAL_SEED).shuffle(_ids)
K = 5; fold_sz = len(_ids)//K
oof_accs, oof_f1s = [], []
for k in range(K):
    va_ids = set(_ids[k*fold_sz:(k+1)*fold_sz]); tr_ids = set(_ids)-va_ids
    tr = [r for r in records if r["sample_id"] in tr_ids]
    va = [r for r in records if r["sample_id"] in va_ids]
    # tune tau on this fold's train
    bt, bs = 0.8, 0.0
    for tau in [x/100 for x in range(50,101,5)]:
        m = compute_metrics(tr, make_anti_overclaim(tau, True))
        s = 0.5*m["acc"]+0.5*m["macro_f1"]
        if s>bs: bs,bt = s,tau
    mv = compute_metrics(va, make_anti_overclaim(bt, True))
    oof_accs.append(mv["acc"]); oof_f1s.append(mv["macro_f1"])
    print(f"  fold {k+1}: acc={mv['acc']:.1%}  macro_f1={mv['macro_f1']:.3f}  (tau={bt:.2f}, n_val={len(va)})")
print(f"\n  OOF acc     : {np.mean(oof_accs):.1%} +/- {np.std(oof_accs):.1%}")
print(f"  OOF macro_f1: {np.mean(oof_f1s):.3f} +/- {np.std(oof_f1s):.3f}")


# ==================================================================
# MAIN COMPARISON -- acc + macro-F1 + No-F1 + Unknown-F1 (VAL + FULL)
# ==================================================================
methods = [
    ("pure_qwen_LoRA",            dec_pure_qwen),
    ("pure_z3 (def->qwen)",       dec_pure_z3),
    ("z3_first",                  dec_z3_first),
    ("parallel_hybrid",           dec_parallel_hybrid),
    ("conf_hybrid",               dec_conf_hybrid),
    (f"anti_overclaim(tau={best_tau:.2f})", make_anti_overclaim(best_tau, True)),
    ("anti_overclaim_all",        make_anti_overclaim(best_tau, False)),
]

def _row(name, recs, dec):
    m = compute_metrics(recs, dec)
    return (name, m["acc"], m["macro_f1"], m["weighted_f1"],
            m["per"].get("NO",{}).get("f1",0.0),
            m["per"].get("UNKNOWN",{}).get("f1",0.0))

print("\n" + "="*100)
print("  v16_1 (v16 LoRA) -- RULE-BASED METHODS (NO LGBM) -- full metrics")
print("="*100)
print(f"  {'Method':32s} | {'VAL acc':>8} {'mF1':>6} {'wF1':>6} {'No-F1':>6} {'Unk-F1':>7} | {'FULL acc':>9} {'mF1':>6}")
print("  " + "-"*96)
results = {}
for name, dec in methods:
    _, va, vmf, vwf, vno, vun = _row(name, val_recs, dec)
    _, fa, fmf, fwf, _, _     = _row(name, records, dec)
    results[name] = dict(val_acc=va, val_mf1=vmf, val_wf1=vwf, val_no_f1=vno,
                         val_unk_f1=vun, full_acc=fa, full_mf1=fmf, full_wf1=fwf)
    print(f"  {name:32s} | {va:>7.1%} {vmf:>6.3f} {vwf:>6.3f} {vno:>6.3f} {vun:>7.3f} | {fa:>8.1%} {fmf:>6.3f}")
print("  " + "-"*96)
om = compute_metrics(val_recs, dec_oracle); omf = compute_metrics(records, dec_oracle)
print(f"  {'ORACLE (upper bound)':32s} | {om['acc']:>7.1%} {om['macro_f1']:>6.3f} {'':>6} {'':>7} | {omf['acc']:>8.1%} {omf['macro_f1']:>6.3f}")
print("="*88)

# ==================================================================
# q_idx breakdown (generated has q_idx=1 = mostly Yes/No/Unknown)
# ==================================================================
print("\n  Best rule by q_idx (anti_overclaim):")
best_dec = make_anti_overclaim(best_tau, True)
for qi in sorted(set(r["q_idx"] for r in val_recs)):
    sub = [r for r in val_recs if r["q_idx"]==qi]
    m = compute_metrics(sub, best_dec)
    print(f"    q_idx={qi}: acc={m['acc']:.1%}  macro_f1={m['macro_f1']:.3f}  (n={len(sub)})")

# MC vs YN breakdown
print("\n  Best rule by question type:")
for qt_name, pred in [("MC", lambda r: r["q_type"]=="mc"), ("Yes/No/Unk", lambda r: r["q_type"]!="mc")]:
    sub = [r for r in val_recs if pred(r)]
    if sub:
        m = compute_metrics(sub, best_dec)
        print(f"    {qt_name:12s}: acc={m['acc']:.1%}  macro_f1={m['macro_f1']:.3f}  (n={len(sub)})")

# ==================================================================
# CONFUSION MATRIX on VAL (best rule: anti_overclaim)
# ==================================================================
_bm = compute_metrics(val_recs, best_dec)
_labels = sorted(set(_bm["golds"]))
_conf = {g:{p:0 for p in _labels} for g in _labels}
for p,g in zip(_bm["preds"], _bm["golds"]):
    if p in _conf[g]: _conf[g][p]+=1
    else: _conf[g][p]=_conf[g].get(p,0)+1
print("\n" + "="*70)
print("  Confusion matrix on VAL (anti_overclaim) -- gold(row) x pred(col)")
print("="*70)
_allp = sorted(set(_bm["preds"]) | set(_labels))
print("  " + " "*10 + "".join(f"{p[:7]:>8}" for p in _allp))
for g in _labels:
    row = _conf[g]
    tot = sum(row.values())
    cells = "".join(f"{row.get(p,0):>8}" for p in _allp)
    print(f"  {g:>9} {cells}   n={tot}")

# ==================================================================
# SAVE
# ==================================================================
summary = {
    "meta": {"version":"v16_1_v16_no_lgbm_antioverclaim",
             "ground_tau": best_tau, "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
             "train_samples": len(train_ids), "val_samples": len(val_ids)},
    "methods": results,
    "oracle": {"val_acc": om["acc"], "val_macro_f1": om["macro_f1"],
               "full_acc": omf["acc"], "full_macro_f1": omf["macro_f1"]},
    "anti_overclaim_5fold": {"oof_acc_mean": float(np.mean(oof_accs)),
                             "oof_acc_std": float(np.std(oof_accs)),
                             "oof_macro_f1_mean": float(np.mean(oof_f1s)),
                             "oof_macro_f1_std": float(np.std(oof_f1s))},
}
json.dump(summary, open("/kaggle/working/pipeline_results_16_1.json","w",encoding="utf-8"),
          ensure_ascii=False, indent=2)
print(f"\nSaved: /kaggle/working/pipeline_results_16_1.json")
print("\n" + "="*70 + "\n  v16_1 (v16 LoRA, no LGBM, anti-overclaim) HOAN TAT\n" + "="*70)



  Tuning anti-overclaim GROUND_TAU on TRAIN (counter Yes-bias)
  Best GROUND_TAU = 0.50 (train blended score 0.704)

  5-fold OOF: anti-overclaim (sample-level folds)
  fold 1: acc=73.0%  macro_f1=0.639  (tau=0.85, n_val=126)
  fold 2: acc=80.8%  macro_f1=0.649  (tau=0.85, n_val=125)
  fold 3: acc=73.4%  macro_f1=0.576  (tau=0.85, n_val=124)
  fold 4: acc=75.2%  macro_f1=0.705  (tau=0.85, n_val=125)
  fold 5: acc=63.7%  macro_f1=0.534  (tau=0.50, n_val=124)

  OOF acc     : 73.2% +/- 5.5%
  OOF macro_f1: 0.621 +/- 0.060

  v16_1 (v16 LoRA) -- RULE-BASED METHODS (NO LGBM) -- full metrics
  Method                           |  VAL acc    mF1    wF1  No-F1  Unk-F1 |  FULL acc    mF1
  ------------------------------------------------------------------------------------------------
  pure_qwen_LoRA                   |   73.8%  0.605  0.718  0.154   0.083 |    76.8%  0.658
  pure_z3 (def->qwen)              |   69.8%  0.523  0.679  0.143   0.095 |    77.0%  0.622
  z3_first                  

In [17]:
# ==================================================================
# OPTIONAL ABLATION -- LGBM calibrator with 5-fold OOF
# KHONG dung trong final. Chi de bao cao ablation: "learned routing
# khong vuot rule-based, va co mild overfit". Set RUN_LGBM_ABLATION=False
# de skip.
# ==================================================================
RUN_LGBM_ABLATION = True

if RUN_LGBM_ABLATION:
    import numpy as np, random
    FEATURES = ["z3_conf","z3_grounded","z3_definite","z3_spread","z3_qwen_agree",
                "q_type_mc","n_idx_premises","has_free_var","is_converse",
                "n_quantifiers","stmt_len","qwen_sc_conf","qwen_vote_spread",
                "qwen_n_cited","z3_qwen_jaccard"]
    def _mat(recs):
        X = np.array([[r[f] for f in FEATURES] for r in recs], dtype=float)
        y = np.array([r["z3_correct"] for r in recs], dtype=int)
        return X, y
    try:
        import lightgbm as lgb
        mk = lambda: lgb.LGBMClassifier(n_estimators=300, num_leaves=15, learning_rate=0.04,
                       min_child_samples=10, subsample=0.9, colsample_bytree=0.9,
                       random_state=CAL_SEED, verbose=-1)
    except Exception:
        from sklearn.ensemble import GradientBoostingClassifier
        mk = lambda: GradientBoostingClassifier(n_estimators=300, max_depth=3,
                       learning_rate=0.04, random_state=CAL_SEED)

    # 5-fold OOF accuracy of LGBM-routed decision
    _ids = sorted(set(r["sample_id"] for r in records))
    random.Random(CAL_SEED).shuffle(_ids)
    K=5; fs=len(_ids)//K; fold_acc=[]
    for k in range(K):
        va_ids=set(_ids[k*fs:(k+1)*fs]); tr_ids=set(_ids)-va_ids
        tr=[r for r in records if r["sample_id"] in tr_ids]
        va=[r for r in records if r["sample_id"] in va_ids]
        Xtr,ytr=_mat(tr); clf=mk(); clf.fit(Xtr,ytr)
        p=clf.predict_proba(_mat(va)[0])[:,1]
        # tune tau on this fold train
        ptr=clf.predict_proba(Xtr)[:,1]; bt,ba=0.5,0
        for tau in [x/100 for x in range(20,90,5)]:
            c=sum(1 for j,r in enumerate(tr) if str(r["z3_answer"] if (r["z3_definite"] and ptr[j]>=tau) else r["qwen_answer"]).upper()==str(r["gold"]).upper())
            a=c/len(tr)
            if a>ba: ba,bt=a,tau
        c=sum(1 for j,r in enumerate(va) if str(r["z3_answer"] if (r["z3_definite"] and p[j]>=bt) else r["qwen_answer"]).upper()==str(r["gold"]).upper())
        fold_acc.append(c/len(va))
        print(f"  LGBM fold {k+1}: OOF acc={c/len(va):.1%} (tau={bt:.2f})")
    print(f"\n  LGBM 5-fold OOF: {np.mean(fold_acc):.1%} +/- {np.std(fold_acc):.1%}")
    print(f"  -> So sanh voi rule-based z3_first/anti_overclaim o cell tren.")
    print(f"  -> Neu OOF khong vuot rule + std cao => confirm: bo LGBM khoi final.")
else:
    print("LGBM ablation skipped.")


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


  LGBM fold 1: OOF acc=77.0% (tau=0.45)
  LGBM fold 2: OOF acc=82.4% (tau=0.25)


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


  LGBM fold 3: OOF acc=79.0% (tau=0.35)
  LGBM fold 4: OOF acc=79.2% (tau=0.30)
  LGBM fold 5: OOF acc=75.0% (tau=0.50)

  LGBM 5-fold OOF: 78.5% +/- 2.5%
  -> So sanh voi rule-based z3_first/anti_overclaim o cell tren.
  -> Neu OOF khong vuot rule + std cao => confirm: bo LGBM khoi final.


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [19]:
# ==================================================================
# UTILS -- simple accuracy helper for Stage 5
# ==================================================================
def accuracy(recs, decision_fn, gold_key="gold"):
    """
    Return (acc, correct, total).
    decision_fn can be:
      - a function: lambda r -> answer
      - a string column name: "qwen_answer", "z3_answer", "final_answer"
    """
    correct = 0
    total = 0

    for r in recs:
        if gold_key not in r or r[gold_key] is None:
            continue

        gold = _U(r.get(gold_key, ""))

        if callable(decision_fn):
            pred = decision_fn(r)
        else:
            pred = r.get(decision_fn, "")

        pred = _U(pred)

        if not gold:
            continue

        total += 1
        if pred == gold:
            correct += 1

    acc = correct / total if total else 0.0
    return acc, correct, total

In [23]:
# ==================================================================
# STAGE 5 -- Sample Review + Per-class + Train/Val Gap + Faithfulness
# Reads records / train_recs / val_recs and all deciders from Stage 4.
# Loads logic_samples fresh (same SEED -> same shuffle) for question text.
# ==================================================================
import random as _rnd, json, os
from collections import defaultdict, Counter

# ---- Reload samples for text lookup (same SEED -> same order as build_features) ----
_raw_path = DATASET_PATH
_raw = json.load(open(_raw_path, encoding="utf-8"))
# Replicate same shuffle load_dataset uses so sample_id lines up
_shuffled = _raw[:]
_rnd.Random(SEED).shuffle(_shuffled)

def _get(sample_id, q_idx):
    """Return (question_text, explanation, premises_nl) for a record."""
    if sample_id >= len(_shuffled):
        return ("(not found)", "", [])
    s = _shuffled[sample_id]
    q   = s.get("questions", [])[q_idx] if q_idx < len(s.get("questions", [])) else "(not found)"
    exp = s.get("explanation", [])[q_idx] if q_idx < len(s.get("explanation", [])) else ""
    nls = s.get("premises-NL", [])
    return q, exp, nls

# ==================================================================
# (A) Per-class accuracy -- TRAIN vs VAL for each source
# ==================================================================

def _cls_acc(recs, field, target_cls):
    sub = [r for r in recs if str(r["gold"]).upper() == target_cls.upper()]
    if not sub: return (0.0, 0, 0)
    c = sum(1 for r in sub if str(r[field]).upper() == target_cls.upper())
    return (100*c/len(sub), c, len(sub))

all_cls = sorted(set(str(r["gold"]).upper() for r in records))

print("\n" + "="*80)
print("  STAGE 5 -- Per-class accuracy: Z3 vs Qwen (TRAIN vs VAL)")
print("="*80)
print(f"  {'Class':>10} | {'Z3 train':>10} {'Z3 val':>10} | {'Qwen train':>12} {'Qwen val':>10} | {'N val':>6}")
print(f"  {'-'*70}")
for cls in all_cls:
    zt  = _cls_acc(train_recs, "z3_answer",   cls)
    zv  = _cls_acc(val_recs,   "z3_answer",   cls)
    qt  = _cls_acc(train_recs, "qwen_answer",  cls)
    qv  = _cls_acc(val_recs,   "qwen_answer",  cls)
    flag = ""
    if cls == "NO" and zv[2] and zv[0] < 60: flag = "  <- Z3 underfit"
    if zt[0] - zv[0] > 15:                   flag = "  <- overfit Z3"
    if qt[0] - qv[0] > 15:                   flag += "  <- overfit Qwen"
    print(f"  {cls:>10} | {zt[0]:>9.0f}% {zv[0]:>9.0f}% | {qt[0]:>11.0f}% {qv[0]:>9.0f}% | {zv[2]:>6}{flag}")
print("="*80)

# ==================================================================
# (B) Train vs Val gap -- all methods
# ==================================================================
print("\n" + "="*80)
print("  Train vs Val gap per method (GAP = train_acc - val_acc)")
print("  Gap < 5pp = OK | 5-10pp = mild overfit | >10pp = heavy overfit")
print("="*80)
print(f"  {'Method':36s} | {'Train':>9} | {'Val':>9} | {'GAP':>7}")
print(f"  {'-'*65}")

_methods = [
    ("pure_qwen_LoRA",          dec_pure_qwen),
    ("pure_z3 (definite->qwen)",dec_pure_z3),
    ("z3_first",                dec_z3_first),
    ("parallel_hybrid",         dec_parallel_hybrid),
    ("conf_hybrid",             dec_conf_hybrid),
    ("oracle_upper_bound",      dec_oracle),
]
for name, dec in _methods:
    tr = accuracy(train_recs, dec); va = accuracy(val_recs, dec)
    gap = tr[0]*100 - va[0]*100
    flag = ("  <- OK" if gap < 5 else ("  <- mild" if gap < 10 else "  <- OVERFIT"))
    print(f"  {name:36s} | {tr[0]:>8.1%} | {va[0]:>8.1%} | {gap:>+6.1f}pp{flag}")

# calibrator (indexed) -- ablation only, skip if not available
if "p_tr" in globals() and "best_tau" in globals() and "cal_val_c" in globals():
    _cal_tr = sum(
        1 for j, r in enumerate(train_recs)
        if _U(r["z3_answer"] if (r.get("z3_definite", 0) and p_tr[j] >= best_tau) else r["qwen_answer"])
        == _U(r["gold"])
    ) / len(train_recs)

    _cal_va = cal_val_c / len(val_recs)
    _gap_cal = _cal_tr * 100 - _cal_va * 100
    _flag_cal = "  <- OK" if _gap_cal < 5 else ("  <- mild" if _gap_cal < 10 else "  <- OVERFIT")

    print(
        f"  {'CALIBRATOR LGBM (ablation only)':36s} | "
        f"{_cal_tr*100:7.1f}% | {_cal_va*100:7.1f}% | {_gap_cal:+6.1f}pp{_flag_cal}"
    )
else:
    print(
        f"  {'CALIBRATOR LGBM (ablation skipped)':36s} | "
        f"{'N/A':>7s} | {'N/A':>7s} | {'N/A':>8s}"
    )

print("=" * 80)
# ==================================================================
# (C) Faithfulness metrics (v13.6 unique: z3_qwen_jaccard, qwen_n_cited)
# ==================================================================
print("\n" + "="*70)
print("  Faithfulness metrics on VAL (XAI deliverable)")
print("="*70)
_jac   = [r["z3_qwen_jaccard"] for r in val_recs if r["z3_definite"]]
_cited = [r["qwen_n_cited"]   for r in val_recs]
_conf  = [r["qwen_sc_conf"]   for r in val_recs]

if _jac:
    import numpy as np
    print(f"  z3_qwen_jaccard (semantic agreement, Z3-definite only, n={len(_jac)}):")
    print(f"    mean={np.mean(_jac):.3f}  median={np.median(_jac):.3f}  "
          f"p25={np.percentile(_jac,25):.3f}  p75={np.percentile(_jac,75):.3f}")
    _j_hi = sum(1 for j in _jac if j >= 0.5)
    print(f"    jaccard>=0.5 (strong agreement): {_j_hi}/{len(_jac)} = {100*_j_hi/len(_jac):.0f}%")
    print(f"    jaccard==0   (no overlap):       "
          f"{sum(1 for j in _jac if j==0)}/{len(_jac)}")
print(f"\n  qwen_n_cited (premises cited in CoT):  "
      f"mean={sum(_cited)/len(_cited):.2f}  0-cite={sum(1 for c in _cited if c==0)}/{len(_cited)}")
print(f"  qwen_sc_conf (BoN self-consistency):   "
      f"mean={sum(_conf)/len(_conf):.3f}  conf==1.0={sum(1 for c in _conf if c>=0.999)}/{len(_conf)}")

# ==================================================================
# (D) Sample review -- 5 TRAIN + 5 VAL with full detail
# ==================================================================
_rs = _rnd.Random(42)

def _print_samples(title, recs, n=5):
    print("\n" + "="*70)
    print(f"  {title}")
    print("="*70)
    chosen = _rs.sample(recs, min(n, len(recs)))
    for i, r in enumerate(chosen, 1):
        q, exp, nls = _get(r["sample_id"], r["q_idx"])
        _z_ok = str(r["z3_answer"]).upper() == str(r["gold"]).upper()
        _q_ok = str(r["qwen_answer"]).upper() == str(r["gold"]).upper()
        # Calibrator decision for this record
        # Calibrator decision for this record -- ablation only
        _has_cal = ("p_full" in globals()) and ("best_tau" in globals())

        if _has_cal:
            _idx = records.index(r)
            _cal_pred = r["z3_answer"] if (
                r.get("z3_definite", 0) and p_full[_idx] >= best_tau
            ) else r["qwen_answer"]
            _c_ok = _U(_cal_pred) == _U(r["gold"])
        else:
            _cal_pred = None
            _c_ok = None
        print(f"\n  [{i}]  gold={r['gold']:>8s}  q_type={r['q_type']}")
        print(f"       Q : {q[:105]}")
        # Show 2 premises
        for pi, p in enumerate(nls[:2]):
            print(f"       P{pi+1}: {p[:90]}")
        if len(nls) > 2:
            print(f"       ... ({len(nls)-2} more premises)")
        print(f"       Z3   : {r['z3_answer']:>8s} (def={r['z3_definite']}  conf={r['z3_conf']:.2f}  grnd={r['z3_grounded']:.2f})  {'OK' if _z_ok else 'WRONG'}")
        print(f"       Qwen : {r['qwen_answer']:>8s} (sc_conf={r.get('qwen_sc_conf',0):.2f}  spread={r.get('qwen_vote_spread',0)}  cited={r.get('qwen_n_cited',0)})  {'OK' if _q_ok else 'WRONG'}")
        if _has_cal:
            print(f"       Cal  : {_cal_pred:>8s} (tau={best_tau:.2f})  {'OK' if _c_ok else 'WRONG'}")
        else:
            print(f"       Cal  : {'N/A':>8s} (LGBM ablation skipped)")
        print(f"       Jacc : {r.get('z3_qwen_jaccard',0):.3f}  agree={r['z3_qwen_agree']}")
        print(f"       Exp  : {exp[:120]}...")

_print_samples("5 mau TRAIN -- da hoc (z3+qwen co the memorize)", train_recs)
_print_samples("5 mau VAL   -- chua thay (so nay moi tin duoc)", val_recs)

# ==================================================================
# (E) Both-wrong analysis (model capacity limit on VAL)
# ==================================================================
_both_wrong  = [r for r in val_recs if not r["z3_correct"] and not r["qwen_correct"]]
_z3_only     = [r for r in val_recs if r["z3_correct"]  and not r["qwen_correct"]]
_qwen_only   = [r for r in val_recs if r["qwen_correct"] and not r["z3_correct"]]
_both_right  = [r for r in val_recs if r["z3_correct"]  and r["qwen_correct"]]

print("\n" + "="*70)
print("  Disagreement breakdown on VAL")
print("="*70)
print(f"  Both right       : {len(_both_right):4d}  ({100*len(_both_right)/len(val_recs):.1f}%)")
print(f"  Only Z3 right    : {len(_z3_only):4d}  ({100*len(_z3_only)/len(val_recs):.1f}%)  <- Z3 co gia tri")
print(f"  Only Qwen right  : {len(_qwen_only):4d}  ({100*len(_qwen_only)/len(val_recs):.1f}%)  <- Qwen co gia tri")
print(f"  Both wrong       : {len(_both_wrong):4d}  ({100*len(_both_wrong)/len(val_recs):.1f}%)  <- model capacity limit, khong routing nao cuu duoc")
print(f"  Max routing gain : {len(_z3_only)+len(_qwen_only)} questions  "
      f"({100*(len(_z3_only)+len(_qwen_only))/len(val_recs):.1f}%)")
print(f"  Oracle achieves  : {len(_both_right)+len(_z3_only)+len(_qwen_only)}/{len(val_recs)} "
      f"({100*(len(_both_right)+len(_z3_only)+len(_qwen_only))/len(val_recs):.1f}%) -- matches oracle_upper_bound above")

if _both_wrong:
    print(f"\n  3 mau 'both wrong' (chi tiet):")
    for i, r in enumerate(_rs.sample(_both_wrong, min(3, len(_both_wrong))), 1):
        q, exp, nls = _get(r["sample_id"], r["q_idx"])
        print(f"\n  [{i}] gold={r['gold']:>8s}  z3={r['z3_answer']:>8s}  qwen={r['qwen_answer']:>8s}  q_type={r['q_type']}")
        print(f"       Q   : {q[:100]}")
        print(f"       Exp : {exp[:100]}...")
        print(f"       z3_conf={r['z3_conf']:.2f}  qwen_sc_conf={r.get('qwen_sc_conf',0):.2f}  jaccard={r.get('z3_qwen_jaccard',0):.3f}")

# ==================================================================
# (F) Save supplementary analysis
# ==================================================================
_supp = {
    "meta": {"version": "v16_1_v16_stage5_analysis",
             "timestamp": __import__("time").strftime("%Y-%m-%d %H:%M:%S"),
             "n_train": len(train_recs), "n_val": len(val_recs)},
    "per_class_val": {
        cls: {"z3_acc": _cls_acc(val_recs,"z3_answer",cls)[0],
              "qwen_acc": _cls_acc(val_recs,"qwen_answer",cls)[0],
              "n": _cls_acc(val_recs,"z3_answer",cls)[2]}
        for cls in all_cls
    },
    "gap_analysis": {
        name: {"train_acc": accuracy(train_recs,dec)[0],
               "val_acc":   accuracy(val_recs,dec)[0],
               "gap_pp":    accuracy(train_recs,dec)[0]*100 - accuracy(val_recs,dec)[0]*100}
        for name, dec in _methods
    },
    "faithfulness": {
        "jaccard_mean":  float(sum(_jac)/len(_jac)) if _jac else 0,
        "jaccard_ge05":  sum(1 for j in _jac if j>=0.5),
        "cited_mean":    float(sum(_cited)/len(_cited)),
        "scconf_mean":   float(sum(_conf)/len(_conf)),
    },
    "disagreement": {
        "both_right":  len(_both_right),
        "z3_only":     len(_z3_only),
        "qwen_only":   len(_qwen_only),
        "both_wrong":  len(_both_wrong),
    },
}
_supp_path = "/kaggle/working/pipeline_results_16_1_stage5.json"
json.dump(_supp, open(_supp_path,"w",encoding="utf-8"), ensure_ascii=False, indent=2)
print(f"\nSaved: {_supp_path}")

print("\n" + "="*70 + "\n  STAGE 5 HOAN TAT\n" + "="*70)



  STAGE 5 -- Per-class accuracy: Z3 vs Qwen (TRAIN vs VAL)
       Class |   Z3 train     Z3 val |   Qwen train   Qwen val |  N val
  ----------------------------------------------------------------------
           A |        69%        52% |          85%        88% |     25  <- overfit Z3
           B |        54%        17% |          77%        50% |      6  <- overfit Z3  <- overfit Qwen
           C |        53%        56% |          89%       100% |      9
           D |        17%        38% |          83%       100% |      8
          NO |        11%         0% |          32%        11% |      9  <- Z3 underfit  <- overfit Qwen
     UNKNOWN |        53%        54% |          23%         8% |     13  <- overfit Qwen
         YES |        82%        62% |          95%        88% |     56  <- overfit Z3

  Train vs Val gap per method (GAP = train_acc - val_acc)
  Gap < 5pp = OK | 5-10pp = mild overfit | >10pp = heavy overfit
  Method                               |     Train |   